# Projet A1 · Consommation énergétique des bâtiments · ⭐⭐⭐

**Piste « Projets avancés » · Niveau ⭐⭐⭐ Avancé · Pipeline complet façon soutenance**

- **Question métier** : la ville de Seattle veut estimer la consommation d'énergie et les émissions de CO₂ de ses bâtiments non résidentiels **sans** relevé annuel coûteux, à partir de leurs caractéristiques (surface, usage, âge, quartier).
- **Ce qu'on construit** : un pipeline complet — nettoyage, feature engineering, exploration, 4 modèles candidats, **XGBoost réglé par `RandomizedSearchCV`**, interprétation **SHAP**, et une réponse à la question « l'ENERGY STAR score aide-t-il vraiment ? ».
- **Livrable** : ce notebook exécuté + un rapport de 10-15 slides selon `../gabarit-rapport.md` + une phrase de synthèse.

Comment l'utiliser :
- Google Colab (aucun GPU nécessaire ici) ou en local dans l'environnement `projets-avances/requirements.txt`. `Maj + Entrée` cellule après cellule.
- `MODE_RAPIDE = True` (par défaut) : petites grilles, tout tourne en 2-3 minutes sur CPU. `MODE_RAPIDE = False` : recherche complète (≈ 5 min sur Colab).
- Les cellules **« À toi »** sont des exercices : le squelette s'exécute tel quel, la vérification affiche ✅ / ❌, la solution est repliée juste en dessous. Essaie avant d'ouvrir !
- Les cellules **« Rapport »** impriment les chiffres à recopier dans ton rapport.

## 0. Préparation

In [ ]:
import os, io, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODE_RAPIDE = True          # False = recherche d'hyperparamètres complète (plus long)
SEED = 42
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_columns", 30)

try:                        # sur certains Mac, Python ne trouve pas les certificats HTTPS sans ce réglage
    import certifi
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
except ImportError:
    pass

DATA_DIR = Path("data")     # cache local des données (dossier ignoré par git)
DATA_DIR.mkdir(exist_ok=True)
print("MODE_RAPIDE =", MODE_RAPIDE)

In [ ]:
resultats = {}

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = un booléen ou une fonction)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else " — pas encore, relis l'énoncé ou ouvre l'indice."))

def proche(a, b, tol=0.05):
    """Vrai si a est à moins de tol (en relatif) de b."""
    return abs(a - b) <= tol * max(abs(b), 1e-9)

print("Helpers prêts.")

## 1. Contexte et question métier

Depuis 2016, Seattle oblige les bâtiments de plus de 20 000 pieds carrés à **déclarer chaque année** leur consommation d'énergie (programme *Energy Benchmarking*). Ces relevés coûtent cher à collecter et arrivent avec un an de retard. La ville aimerait **prédire** la consommation d'énergie (`siteenergyuse_kbtu`) et les émissions de gaz à effet de serre (`totalghgemissions`) des bâtiments **non résidentiels** à partir de ce qu'on connaît sans relevé : surface, nombre d'étages, année de construction, type d'usage, quartier.

À qui ça sert ? Au service climat de la ville, pour cibler les bâtiments à rénover en priorité et pour estimer les émissions des bâtiments qui n'ont pas encore déclaré. Un « bon résultat » : un modèle dont l'erreur typique est nettement plus petite que la simple médiane, et qui reste **explicable** — la ville doit pouvoir dire pourquoi un bâtiment est jugé énergivore.

**Métrique choisie** : les consommations s'étalent sur plusieurs ordres de grandeur (de 10⁴ à 10⁸ kBtu). On modélise donc le **logarithme** de la cible et on suit le **RMSE en log** (une erreur de 0,3 en log ≈ 35 % d'écart relatif), complété par le **MAE en unités réelles** (lisible pour un décideur) et le **R²**. Question bonus, comme dans le projet OpenClassrooms « P4 » : l'**ENERGY STAR score** (un indicateur lui aussi coûteux, manquant pour un tiers des bâtiments) améliore-t-il la prédiction au point de justifier son calcul ?

## 2. Les données

Source : [Seattle Open Data — Building Energy Benchmarking](https://data.seattle.gov/d/teqw-tu6e) (licence publique, API Socrata). On charge l'année 2016 (≈ 3 300 bâtiments, tous types confondus), on met le fichier en cache dans `data/`, et si le réseau manque on utilise un échantillon de 300 bâtiments intégré à la cellule suivante.

### Dictionnaire des variables

| Variable | Sens | Unité / valeurs |
|---|---|---|
| `osebuildingid`, `buildingname` | identifiant et nom du bâtiment | — |
| `buildingtype` | grande famille : `NonResidential`, `Multifamily …`, `School`, `Hospital`, `University` | catégorie |
| `epapropertytype`, `largestpropertyusetype` | type de propriété EPA et **usage principal** (bureau, hôtel, entrepôt…) | catégorie |
| `largestpropertyusetypegfa` | surface occupée par l'usage principal | pieds carrés (sq ft) |
| `neighborhood` | quartier | catégorie |
| `yearbuilt` | année de construction | année |
| `numberoffloors`, `numberofbuildings` | nombre d'étages, nombre de bâtiments de la propriété | entier |
| `propertygfatotal` | surface totale (GFA) | sq ft |
| `propertygfaparking`, `propertygfabuildings` | surface de parking / de bâtiments | sq ft |
| `energystarscore` | score ENERGY STAR (1-100, 100 = très performant), **souvent manquant** | score |
| **`siteenergyuse_kbtu`** | **cible 1** : énergie consommée sur site en 2016 | kBtu (1 kBtu ≈ 0,29 kWh) |
| **`totalghgemissions`** | **cible 2** : émissions de gaz à effet de serre | tonnes CO₂e |
| `latitude`, `longitude` | position | degrés |

In [ ]:
DONNEES_SECOURS = """osebuildingid,buildingname,buildingtype,epapropertytype,largestpropertyusetype,largestpropertyusetypegfa,neighborhood,yearbuilt,numberoffloors,numberofbuildings,propertygfatotal,propertygfaparking,propertygfabuildings,energystarscore,siteenergyuse_kbtu,totalghgemissions,latitude,longitude
26695,THE CHURCH IN SEATTLE,NonResidential,Worship Facility,Worship Facility,27500,NORTHWEST,1978,2,1,27500,0,27500,60,5.7712e+05,14.4,47.678,-122.33
15,HOTEL MONACO,NonResidential,Hotel,Hotel,133884,DOWNTOWN,1969,11,1,153163,19279,133884,41,1.6117e+07,708.6,47.607,-122.33
20433,West Seattle Bowl / Highstrike Grill,NonResidential,Other - Recreation,Other - Recreation,26241,SOUTHWEST,1948,1,1,41521,13124,28397,,3.3436e+06,98.1,47.563,-122.38
1280,FILSON WORLD HEADQUARTERS LP,NonResidential,Mixed Use Property,Office,26044,GREATER DUWAMISH,1921,3,1,66480,13200,53280,3,7.071e+06,41,47.587,-122.33
340,PIONEER BUILDING,NonResidential,Office,Office,82355,DOWNTOWN,1900,6,1,89355,0,89355,66,4.1782e+06,27.1,47.602,-122.33
20029,PHI DELTA THETA,NonResidential,Residence Hall/Dormitory,Residence Hall/Dormitory,21499,NORTHEAST,1921,3,1,21499,0,21499,85,6.0513e+05,3.5,47.663,-122.31
24435,10TH AVE BLDG,NonResidential,Office,Office,27600,EAST,1907,2,1,36000,0,36000,95,1.0462e+06,17.1,47.613,-122.32
40028,"UW- SANDPOINT BUILDING # 5 (5A,5B,5C,5D)",University,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,349953,NORTHEAST,1940,1,1,384772,0,384772,69,1.5207e+07,380.5,47.685,-122.26
465,1000 DENNY WAY BUILDING,NonResidential,Office,Office,155985,LAKE UNION,1929,8,1,287819,0,287819,,4.8407e+07,292.6,47.619,-122.34
50,LAWTON ELEMENTARY SCHOOL (SPS-DISTRICT),School,K-12 School,K-12 School,54986,MAGNOLIA / QUEEN ANNE,1990,2,1,54986,0,54986,97,1.6503e+06,30,47.657,-122.39
26532,KALBERG BUILDING,NonResidential,Other - Mall,Other - Mall,20760,NORTHEAST,1928,2,1,20760,0,20760,,1.6886e+06,46.1,47.662,-122.31
21772,ARTISTS LOFTS,NonResidential,Office,Office,10292,BALLARD,1999,4,1,28844,0,28844,,9.1072e+05,5.3,47.658,-122.37
470,401 TERRY AVE (INSTITUTE FOR SYSTEM BIOLOGY),NonResidential,Laboratory,Laboratory,176171,LAKE UNION,2003,4,1,258505,82334,176171,,3.3714e+07,824.5,47.623,-122.34
24030,KING PLAZA II,NonResidential,Retail Store,Retail Store,30000,GREATER DUWAMISH,2002,2,1,54811,17471,37340,100,1.2675e+06,7.4,47.539,-122.28
23934,Prologis Park Seattle 2,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,53550,GREATER DUWAMISH,1960,1,1,53550,0,53550,11,4.5902e+06,172.5,47.556,-122.34
20986,ORCAS BUILDING - KING COUNTY,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,25000,GREATER DUWAMISH,1968,1,1,27680,0,27680,,1.7784e+06,63.9,47.55,-122.32
50194,BALLARD SPACE,NonResidential,Office,Office,34350,BALLARD,2016,4,1,37100,0,37100,,,,47.667,-122.38
21783,601 S Alaska Building,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,18373,GREATER DUWAMISH,1989,1,1,21101,0,21101,,6.7577e+05,27.7,47.561,-122.33
19902,SEMKKANN LLC,NonResidential,Retail Store,Retail Store,13940,DOWNTOWN,1925,1,1,24379,1263,23116,,4.3952e+05,10.8,47.618,-122.36
723,NORTH COAST ELECTRIC ETC,NonResidential,Distribution Center,Distribution Center,44300,GREATER DUWAMISH,1970,1,1,84420,0,84420,55,2.8483e+06,16.5,47.581,-122.32
23672,2324 EASTLAKE,NonResidential,Office,Office,34160,LAKE UNION,1986,4,1,57432,21906,35526,,2.5205e+06,14.6,47.64,-122.33
20613,QFC WAREHOUSE,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,20400,GREATER DUWAMISH,1951,1,1,24000,0,24000,,1.257e+06,46,47.57,-122.33
785,Airport Way Ctr - Bldg A,NonResidential,Other,Other,50036,GREATER DUWAMISH,1944,4,1,99122,0,99122,,1.4469e+07,164.3,47.583,-122.32
20707,MALLORY SAFETY & SUPPLY LLC,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,23545,GREATER DUWAMISH,1938,1,1,30545,0,30545,12,1.577e+06,60.8,47.553,-122.34
20976,NWCP BLDG E,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,22320,GREATER DUWAMISH,1977,1,1,22320,0,22320,62,4.3907e+05,2.5,47.554,-122.32
24026,U-HAUL,NonResidential,Self-Storage Facility,Self-Storage Facility,23250,GREATER DUWAMISH,1955,1,1,23100,0,23100,,8.637e+05,13,47.545,-122.29
211,NORTH SEATTLE COLLEGE - CAMPUS,University,College/University,College/University,653167,NORTHWEST,1970,2,11,694072,111625,582447,,3.5891e+07,323.4,47.699,-122.33
329,1700 SEVENTH BUILDING,NonResidential,Office,Office,542461,DOWNTOWN,2000,23,1,747747,205076,542671,90,3.0985e+07,326.1,47.614,-122.34
20525,KUSAK CUT GLASS WORKS- DEMOLISHED/INACTIVE,NonResidential,Distribution Center,Distribution Center,20736,GREATER DUWAMISH,1954,1,0,23236,0,23236,,4.3387e+05,12.7,47.586,-122.3
412,MIKEN BLDG.,NonResidential,Office,Office,41958,DOWNTOWN,1920,8,1,60660,0,60660,,2.5861e+06,15,47.61,-122.34
21622,WHSE MULTI TENANT,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,16768,GREATER DUWAMISH,1977,1,1,29380,0,29380,33,8.5609e+05,5,47.539,-122.32
760,TWO PINE,NonResidential,Office,Office,138257,DOWNTOWN,1908,10,1,132998,0,132998,97,3.9745e+06,153.1,47.61,-122.34
77,FOUR POINTS SHERATON SEATTLE CENTER,NonResidential,Hotel,Hotel,78051,MAGNOLIA / QUEEN ANNE,1999,4,1,122942,44891,78051,94,5.3373e+06,168.5,47.625,-122.34
21142,WAREHOUSE-STUDIO,NonResidential,Other,Other,30730,LAKE UNION,1969,1,1,30730,0,30730,,3.0505e+06,71.1,47.651,-122.36
24539,CLOVERDALE BUSINESS PARK (BLDG E),NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,27850,GREATER DUWAMISH,1979,1,1,27420,0,27420,55,1.1763e+06,38.4,47.525,-122.33
354,FOURTH & MADISON ( IDX TOWER),NonResidential,Office,Office,765351,DOWNTOWN,2002,37,1,1052469,164420,888049,93,4.1214e+07,239,47.605,-122.33
24376,SE Seattle Community Health Center,NonResidential,Medical Office,Medical Office,19987,SOUTHEAST,1990,2,1,38352,11082,27270,36,2.1089e+06,12.2,47.564,-122.29
107,CITY PLACE IV (Ruby/Dawson),NonResidential,Office,Office,598801,LAKE UNION,2010,12,1,571329,0,571329,91,3.9606e+07,579,47.621,-122.34
20930,QUEEN ANNE ELEMENTARY (SPS-DISTRICT),School,K-12 School,K-12 School,48789,MAGNOLIA / QUEEN ANNE,1925,1,1,72445,0,72445,94,1.2983e+06,28.4,47.638,-122.35
28019,OUR LADY OF FATIMA CHURCH & RECTORY,NonResidential,Worship Facility,Worship Facility,31805,MAGNOLIA / QUEEN ANNE,1953,2,1,31805,0,31805,10,1.6417e+06,67.8,47.647,-122.4
21894,KING MANUFACTURING,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,35289,GREATER DUWAMISH,2007,1,1,35289,0,35289,63,8.7642e+05,23.3,47.52,-122.32
731,SALVATION ARMY- demolished,NonResidential,Mixed Use Property,Non-Refrigerated Warehouse,58605,GREATER DUWAMISH,1930,2,3,75020,18860,56160,,3.6397e+06,21.1,47.594,-122.33
23204,MAPLE ELEMENTARY SCHOOL(SPS-DISTRICT),School,K-12 School,K-12 School,53234,GREATER DUWAMISH,1971,1,1,50546,0,50546,94,1.1551e+06,10.9,47.558,-122.32
49872,MERCEDES BENZ OF SEATTLE 2025,NonResidential,Vehicle Dealership,Vehicle Dealership,24520,GREATER DUWAMISH,2014,2,1,56988,0,56988,,5.4417e+05,15.8,47.584,-122.32
257,Harbor Island Studios - KING COUNTY,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,79600,GREATER DUWAMISH,1916,4,1,79600,0,79600,34,2.4114e+06,14,47.575,-122.36
551,RICHMARK PRINTING,NonResidential,Mixed Use Property,Other,28442,EAST,1927,2,1,75844,19928,55916,,3.6842e+06,34.5,47.616,-122.32
82,WA DEPT OF SOCIAL & HEALTH SERVICES (DSHS),NonResidential,Office,Office,54984,CENTRAL,1962,2,1,54984,0,54984,71,3.3752e+06,85.5,47.608,-122.31
603,1201 EASTLAKE AVE E (ZymoGenetics),NonResidential,Mixed Use Property,Other,106003,LAKE UNION,1920,3,1,169785,63782,106003,,1.8846e+07,374.1,47.631,-122.33
23622,PORT OF SEATTLE- FISHERMAN'S TERMINAL  (CAMPUS),NonResidential,Other,Other,260241,MAGNOLIA / QUEEN ANNE,1955,1,25,275701,0,275701,,3.4013e+07,533.3,47.656,-122.38
23158,BRIGGS TECHNOLOGIES INC,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,25840,DOWNTOWN,1956,2,0,25840,0,25840,92,3.1353e+05,1.8,47.598,-122.32
50212,Conservatory Campus,NonResidential,Other - Recreation,Other - Recreation,23445,EAST,1912,1,1,23445,0,23445,,5.9762e+06,257.2,47.632,-122.32
289,FRANTZ H COE ELEMENTARY SCHOOL (SPS-DISTRICT),School,K-12 School,K-12 School,75214,MAGNOLIA / QUEEN ANNE,2003,3,1,75214,0,75214,88,2.3343e+06,41.9,47.641,-122.37
753,THE WESTIN BUILDING EXCHANGE (Office),NonResidential,Data Center,Data Center,218997,DOWNTOWN,1981,33,1,429405,0,429405,98,2.7468e+08,1600.2,47.614,-122.34
600,MERLINO FOODS,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,98480,GREATER DUWAMISH,1956,1,1,98480,0,98480,65,2.8316e+06,36.6,47.566,-122.33
815,FIFTH & YESLER BLDG.,NonResidential,Office,Office,295494,DOWNTOWN,2009,17,1,413264,138098,275166,91,1.3631e+07,79.1,47.602,-122.33
23671,AREIS BUILDING,NonResidential,Office,Office,50333,LAKE UNION,1959,4,1,50333,18036,32297,88,1.6021e+06,9.3,47.642,-122.33
19742,NEW CROCODILE (Formerly El Gaucho),NonResidential,Restaurant,Restaurant,31020,DOWNTOWN,1955,2,1,31020,0,31020,,4.97e+06,205.8,47.615,-122.35
21315,1518 5TH AVE OFFICE,NonResidential,Office,Office,25000,DOWNTOWN,1903,3,1,40260,0,40260,,24106,0.1,47.611,-122.34
291,THURGOOD MARSHALL (SPS-DISTRICT),School,K-12 School,K-12 School,65310,CENTRAL,1991,2,1,64414,0,64414,67,2.2952e+06,13.3,47.591,-122.3
729,GOODWILL BUILDING,NonResidential,Distribution Center,Distribution Center,109090,GREATER DUWAMISH,1987,1,1,111908,0,111908,87,1.5554e+06,32.9,47.588,-122.33
20565,*DEMOLISHED* SAFEWAY STORE # 368,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,25883,MAGNOLIA / QUEEN ANNE,1962,1,1,25159,0,25159,38,8.9053e+06,241.7,47.638,-122.36
24649,CROWN HILL ELEMENTARY CENTER,NonResidential,Mixed Use Property,Other - Education,12000,BALLARD,1926,1,1,42172,0,42172,,3.2758e+06,151.3,47.698,-122.37
20815,OFFICE-SPRAG- inactive demolished,NonResidential,Office,Office,13945,LAKE UNION,1957,2,0,21183,0,21183,99,3.6154e+05,2.1,47.642,-122.33
27381,CROSS AND CROWN CHURCH,NonResidential,Worship Facility,Worship Facility,26373,NORTHEAST,1938,3,1,26373,0,26373,45,9.6424e+05,37.8,47.663,-122.31
26906,PHI GAMMA DELTA FRATERNITY,NonResidential,Residence Hall/Dormitory,Residence Hall/Dormitory,22124,NORTHEAST,1929,3,1,22124,0,22124,82,6.3768e+05,3.7,47.662,-122.31
462,DELRIDGE BUILDING,NonResidential,Office,Office,55593,DELRIDGE NEIGHBORHOODS,1960,3,1,55593,0,55593,55,4.2249e+06,89.7,47.567,-122.36
24711,HEARTWOOD/FOBES 558,NonResidential,Office,Office,32250,DOWNTOWN,1910,5,1,32250,0,32250,92,7.9604e+05,10.6,47.597,-122.33
21447,224 WESTLAKE,NonResidential,Office,Office,22341,LAKE UNION,1926,4,1,35780,5225,30555,77,1.9128e+06,11.1,47.621,-122.34
22585,OFFICE BUILDING WITH CARETAKER UNIT,NonResidential,Office,Office,25184,MAGNOLIA / QUEEN ANNE,2007,4,1,30152,5680,24472,96,8.4504e+05,4.9,47.66,-122.39
22577,WAREHOUSE,NonResidential,Distribution Center,Distribution Center,18229,MAGNOLIA / QUEEN ANNE,1960,2,1,36498,0,36498,89,5.2001e+05,14.5,47.656,-122.37
370,METROPOLITAN PARK I  (WEST) OFFICE BLDG,NonResidential,Office,Office,348270,DOWNTOWN,1980,18,1,413715,64660,349055,79,2.4225e+07,140.5,47.616,-122.33
804,NORTHGATE NORTH,NonResidential,Parking,Parking,321828,NORTHWEST,2000,5,2,705535,367447,338088,,1.1077e+07,86.7,47.709,-122.33
577,NORTHWAY WEST BUILDING,NonResidential,Office,Office,86483,NORTHWEST,1981,6,1,86483,0,86483,56,6.3612e+06,36.9,47.707,-122.33
630,THE REEDO BUILDING,NonResidential,Office,Office,43122,DOWNTOWN,1904,4,1,66842,0,66842,31,5.6086e+06,134.1,47.597,-122.33
24488,HAWTHORNE HILLS PROF CENTER,NonResidential,Office,Office,20317,NORTHEAST,1967,2,1,20317,0,20317,83,6.2099e+05,3.6,47.669,-122.28
774,PACIFIC BUILDING,NonResidential,Office,Office,227556,DOWNTOWN,1970,22,1,227556,98556,129000,95,8.4934e+06,49.3,47.604,-122.33
335,DENNY BUILDING,NonResidential,Office,Office,161000,DOWNTOWN,1968,12,1,190642,0,190642,91,8.5803e+06,223.7,47.616,-122.34
24102,INGERSOLL - RAND 1,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,15400,GREATER DUWAMISH,1966,1,1,20520,0,20520,20,1.1679e+06,40.2,47.579,-122.33
23928,PLANNED PARENTHOOD OF WESTERN WASHINGTON,NonResidential,Office,Office,25500,CENTRAL,1999,3,1,31357,0,31357,55,1.9651e+06,20.4,47.617,-122.31
374,3005 1ST AVE BUILDING,NonResidential,Mixed Use Property,Laboratory,42389,DOWNTOWN,1980,4,1,109347,39537,69810,,8.2182e+06,82.3,47.618,-122.35
21149,THE WESLEY AT CREMONA,University,Multifamily LR (1-4),Multifamily Housing,22625,MAGNOLIA / QUEEN ANNE,2004,4,1,24501,3752,20749,91,9.588e+05,26.8,47.649,-122.36
633,83 S KING STREET,NonResidential,Office,Office,183916,DOWNTOWN,1904,7,1,204504,0,204504,100,9.7229e+06,56.4,47.598,-122.33
26720,SEATTLE LIGHTING FIXTURE COMPANY,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,26416,DOWNTOWN,1906,4,1,43484,0,43484,18,1.567e+06,9.1,47.6,-122.33
438,1101 MADISON TOWER,NonResidential,Medical Office,Medical Office,270384,EAST,1992,14,1,690158,407795,282363,58,2.1157e+07,122.7,47.609,-122.32
767,1015 THIRD AVENUE BLDG - EXPEDITORS INTERNATIONAL,NonResidential,Office,Office,145491,DOWNTOWN,1998,13,1,293576,74481,219095,25,1.7074e+07,99,47.606,-122.33
716,WESTERN BUILDING,NonResidential,Office,Office,74504,DOWNTOWN,1910,6,1,86204,0,86204,89,3.2419e+06,66.3,47.602,-122.34
21784,GOLDEN GRAIN MACARONI,NonResidential,Refrigerated Warehouse,Refrigerated Warehouse,45594,GREATER DUWAMISH,1914,2,1,45594,0,45594,75,1.3513e+06,7.8,47.561,-122.33
49940,VIRGINIA MASON - Central Pavilion,Hospital,Hospital (General Medical & Surgical),Hospital (General Medical & Surgical),374466,EAST,1920,8,1,374466,0,374466,98,7.3155e+07,3640.9,47.61,-122.33
617,PACIFIC COMMERCIAL BUILDING,NonResidential,Office,Office,42156,DOWNTOWN,1900,5,1,54805,0,54805,74,3.5014e+06,56.4,47.6,-122.33
658,505 UNION STATION,NonResidential,Office,Office,303312,DOWNTOWN,2000,11,1,312512,0,312512,,2.5747e+07,155.4,47.599,-122.33
41,2746 NE 45TH ST - PUBLIC STORAGE,NonResidential,Self-Storage Facility,Self-Storage Facility,26225,NORTHEAST,1955,2,1,26225,0,26225,,3.1836e+05,3.6,47.662,-122.3
45927,Research & Training Building (HARBORVIEW) -,Hospital,Laboratory,Laboratory,181930,EAST,2000,8,1,181930,0,181930,,5.3166e+07,2641.8,47.604,-122.32
22454,GOODWILL BALLARD,NonResidential,Retail Store,Retail Store,23067,BALLARD,1967,1,1,22420,0,22420,82,1.4043e+06,30.6,47.675,-122.37
23896,LONG PAINTING,NonResidential,Distribution Center,Distribution Center,15000,GREATER DUWAMISH,1990,1,1,29619,0,29619,50,1.2803e+06,41.7,47.529,-122.33
803,200 SW MICHIGAN,NonResidential,Office,Office,81526,GREATER DUWAMISH,1929,3,1,85126,0,85126,1,3.7952e+07,252.5,47.541,-122.34
859,GEORGETOWN CENTER BLDG. 1 & BLDG. 3 - BLDG B,NonResidential,Retail Store,Retail Store,74178,GREATER DUWAMISH,1958,1,2,74178,0,74178,,5.4022e+06,130.7,47.549,-122.32
739,LAKEVIEW AT FREMONT,NonResidential,Office,Office,105206,LAKE UNION,2008,4,1,202623,88855,113768,73,8.8735e+06,51.5,47.649,-122.35
27982,OFFICE BUILDING,NonResidential,Office,Office,14256,NORTHWEST,1982,3,1,25111,0,25111,77,9.5435e+05,24.3,47.699,-122.34
22142,SAFEWAY #219,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,25946,GREATER DUWAMISH,1953,1,1,26092,0,26092,40,7.6194e+06,167.6,47.537,-122.28
543,SEATTLE DISTRIBUTION CENTER BLDG A,NonResidential,Distribution Center,Distribution Center,124423,GREATER DUWAMISH,1967,1,1,124472,0,124472,62,5.4648e+06,235.5,47.542,-122.33
26,KING COUNTY COURTHOUSE,NonResidential,Courthouse,Courthouse,537150,DOWNTOWN,1916,10,1,540360,0,540360,76,4.4984e+07,1234.5,47.603,-122.33
21550,BAYVIEW  BUILDING,NonResidential,Office,Office,45864,MAGNOLIA / QUEEN ANNE,1930,3,1,45864,0,45864,94,2.5906e+06,114.6,47.62,-122.36
528,BEDROSIANS TILE & STONE,NonResidential,Retail Store,Retail Store,50200,GREATER DUWAMISH,1966,1,1,50200,0,50200,,1.0475e+06,31.6,47.55,-122.33
24089,646 S HOLGATE ST,NonResidential,Retail Store,Retail Store,25880,GREATER DUWAMISH,1944,1,1,25880,0,25880,100,1.6296e+05,0.9,47.587,-122.32
379,3101 WESTERN (FORMERLY AIRBORNE BLDG),NonResidential,Office,Office,158681,DOWNTOWN,1984,8,1,253103,61986,191117,3,2.9949e+07,173.7,47.618,-122.36
163,FIRE STATION 10/FAC/EOC,NonResidential,Other,Other,42755,DOWNTOWN,2006,4,1,61156,0,61156,,9.3634e+06,168.3,47.601,-122.33
656,ARNOLD PAVILLION,NonResidential,Medical Office,Medical Office,200184,EAST,2004,0,1,225982,0,225982,53,2.0561e+07,707,47.61,-122.32
24270,PHINNEY RIDGE LUTHERAN CHURCH,NonResidential,Worship Facility,Worship Facility,41600,NORTHWEST,1951,2,1,41600,0,41600,58,1.1952e+06,30.1,47.684,-122.35
575,NORTHWAY SQUARE EAST,NonResidential,Office,Office,78200,NORTHWEST,1974,5,1,86400,0,86400,53,9.08e+06,288.2,47.707,-122.33
778,101 KING STREET,NonResidential,Office,Office,84481,DOWNTOWN,1910,6,1,95000,13700,81300,83,4.0527e+06,26.8,47.598,-122.33
24589,A2ALLC (COLOR TILE),NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,15000,GREATER DUWAMISH,1926,2,1,20400,0,20400,69,4.3856e+05,11.9,47.579,-122.33
132,ROOSEVELT SQUARE,NonResidential,Other,Other,119146,NORTHEAST,1929,2,1,228385,100070,128315,,4.2536e+06,131,47.676,-122.32
48287,CENTRAL LINK OPERATIONS & MAINTENANCE - Sound Transit,NonResidential,Office,Office,88698,GREATER DUWAMISH,2005,4,1,162157,0,162157,,4.0614e+07,495.3,47.576,-122.32
700,WHOLE FOODS INTERBAY,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,39500,MAGNOLIA / QUEEN ANNE,2008,1,1,64659,0,64659,31,1.2525e+07,333.5,47.637,-122.38
27673,HOTEL 1000,NonResidential,Hotel,Hotel,198400,DOWNTOWN,2006,10,1,392552,22225,370327,22,1.8025e+07,440.1,47.605,-122.34
23712,Fauntleroy Church United Church of Christ,NonResidential,Worship Facility,Worship Facility,28293,SOUTHWEST,1952,3,1,30740,0,30740,,1.6863e+06,60.5,47.521,-122.39
20242,THE COLLEGIANA (UW MED),NonResidential,Hotel,Hotel,20571,NORTHEAST,1930,3,1,20571,0,20571,72,7.1628e+05,4.2,47.66,-122.32
466,VIRGINIA MASON - Buck Pavilion,Hospital,Urgent Care/Clinic/Other Outpatient,Urgent Care/Clinic/Other Outpatient,158738,EAST,1975,8,1,158738,0,158738,,6.1762e+07,1234.8,47.61,-122.33
28081,BROWN BEAR CAR WASH,NonResidential,Office,Office,13600,BALLARD,1988,2,1,23577,0,23577,7,1.8358e+06,37.2,47.656,-122.36
20978,Terreno Lucile,NonResidential,Manufacturing/Industrial Plant,Manufacturing/Industrial Plant,45320,GREATER DUWAMISH,1976,1,1,47105,0,47105,,1.0235e+07,342.6,47.553,-122.32
469,ROSEN BUILDING,NonResidential,Laboratory,Laboratory,54880,LAKE UNION,1928,3,1,60375,0,60375,,1.1594e+07,260.5,47.623,-122.34
759,ODDFELLOWS LODGE - RETAIL,NonResidential,Mixed Use Property,Office,17120,EAST,1908,4,1,76803,0,76803,,3.687e+06,149.4,47.615,-122.32
480,SEATTLE BIOMEDICAL BUILDING,NonResidential,Mixed Use Property,Other,72407,LAKE UNION,2003,5,1,159700,50654,109046,,3.1372e+07,701.9,47.621,-122.34
19926,7TH & BELL- DEMOLISHED,NonResidential,Office,Office,14317,DOWNTOWN,1965,1,1,23752,0,23752,,1.0672e+06,29.6,47.618,-122.34
489,200 WEST THOMAS BUILDING,NonResidential,Office,Office,64752,MAGNOLIA / QUEEN ANNE,1974,5,1,88062,20826,67236,80,3.4156e+06,19.8,47.621,-122.36
281,UNIVERSITY HEIGHTS CENTER,NonResidential,K-12 School,K-12 School,55500,NORTHEAST,1928,2,1,55653,0,55653,98,1.3776e+06,72.7,47.666,-122.31
22810,LAKESIDE SCHOOL,School,K-12 School,K-12 School,23983,NORTHWEST,1930,3,1,23983,0,23983,61,1.1644e+06,32.5,47.731,-122.33
572,BRIGHT HORIZONS,NonResidential,Mixed Use Property,Office,20000,LAKE UNION,1928,1,1,45000,15000,30000,,2.6039e+06,61.9,47.624,-122.33
24581,SODO DELI / BODY ZONE,NonResidential,Other - Entertainment/Public Assembly,Other - Entertainment/Public Assembly,13144,GREATER DUWAMISH,1937,1,1,20975,0,20975,,3.4744e+05,6.2,47.575,-122.33
26994,ARBORETUM COURT II,NonResidential,Mixed Use Property,Parking,11021,EAST,1988,2,1,21036,0,21036,,2.7358e+06,76.5,47.627,-122.29
23503,SALTYS RESTAURANT,NonResidential,Restaurant,Restaurant,19232,SOUTHWEST,1902,2,1,20398,0,20398,,9.3135e+06,329.4,47.587,-122.38
23951,Terreno Denver,NonResidential,Office,Office,24023,GREATER DUWAMISH,1953,1,1,24291,0,24291,98,6.5542e+05,19.2,47.555,-122.32
21,CENTRAL - SEATTLE PUBLIC LIBRARY,NonResidential,Library,Library,364913,DOWNTOWN,2004,11,1,412000,57000,355000,,1.8589e+07,163.3,47.606,-122.33
25572,PETCO AND OTHERS,NonResidential,Retail Store,Retail Store,46000,NORTHEAST,1930,1,1,46059,0,46059,82,1.5827e+06,19,47.661,-122.32
25325,ALL PILGRIMS CHRISTIAN CHURCH,NonResidential,Worship Facility,Worship Facility,26440,EAST,1906,1,1,26440,0,26440,91,7.199e+05,27.6,47.623,-122.32
868,BMW SEATTLE - SHOWROOM & OFFICE,NonResidential,Vehicle Dealership,Vehicle Dealership,48103,GREATER DUWAMISH,2008,2,1,51856,0,51856,,1.9386e+06,11.2,47.594,-122.32
652,WEST SEATTLE CORPORATE CENTER,NonResidential,Office,Office,121885,DELRIDGE NEIGHBORHOODS,1991,5,1,138106,16221,121885,73,8.3818e+06,48.6,47.568,-122.36
40034,SAND POINT BUILDING 29,NonResidential,Office,Office,31845,NORTHEAST,1960,1,1,21931,0,21931,32,3.9472e+06,143.2,47.683,-122.26
19910,KING CO PUBLIC HEALTH,Hospital,Medical Office,Medical Office,26670,DOWNTOWN,1952,4,1,26670,0,26670,51,1.5532e+06,9,47.615,-122.34
612,GLOBE BUILDING,NonResidential,Office,Office,34647,DOWNTOWN,1900,4,1,69817,0,69817,87,2.1781e+06,78.3,47.6,-122.33
49858,NORTHWEST SCHOOL MULTIPURPOSE FACILITY,School,K-12 School,K-12 School,38000,EAST,2014,2,1,35780,0,35780,22,2.9123e+06,48.9,47.614,-122.33
20415,HOLY ROSARY SCHOOL,School,K-12 School,K-12 School,57896,SOUTHWEST,1922,2,1,33462,0,33462,76,3.9872e+06,184.2,47.565,-122.39
226,ROXHILL ELEMENTARY SCHOOL(SPS-DISTRICT),School,K-12 School,K-12 School,46797,DELRIDGE NEIGHBORHOODS,1958,1,1,46797,0,46797,86,2.0759e+06,72.4,47.518,-122.37
722,INDUSTRIAL TRANSFER WHSE,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,129420,GREATER DUWAMISH,1951,3,1,98220,0,98220,12,6.4681e+06,68.4,47.58,-122.33
23313,SWEDISH HOSPITAL MEDICAL CENTER Invex,Hospital,Office,Office,19658,EAST,1956,3,0,21282,0,21282,69,1.2459e+06,64.6,47.608,-122.32
8,WARWICK SEATTLE HOTEL,NonResidential,Hotel,Hotel,123445,DOWNTOWN,1980,18,1,175580,62000,113580,75,1.4173e+07,497.7,47.614,-122.34
25907,SWEDISH CLUB,NonResidential,Other - Entertainment/Public Assembly,Other - Entertainment/Public Assembly,22960,MAGNOLIA / QUEEN ANNE,1961,3,1,22960,0,22960,,2.267e+06,94.6,47.636,-122.34
26657,UNIVERSITY LUTHERAN CHURCH,NonResidential,Worship Facility,Worship Facility,32098,NORTHEAST,1927,3,1,41013,0,41013,,1.0808e+06,36.8,47.665,-122.31
20743,ST ANNE CHURCH & RECTORY,NonResidential,Worship Facility,Worship Facility,26210,MAGNOLIA / QUEEN ANNE,1963,2,1,26210,0,26210,55,1.0867e+06,47.2,47.632,-122.36
26973,NEW CENTRAL HOTEL CONDOMINIUM,NonResidential,Mixed Use Property,Office,24099,DOWNTOWN,1909,3,1,49299,0,49299,,4.7298e+06,188.2,47.597,-122.32
73,SEATTLE HEBREW ACADEMY,School,K-12 School,K-12 School,56072,EAST,1911,3,1,56072,0,56072,76,3.2348e+06,145.4,47.634,-122.31
352,ABRAHAM LINCOLN BUILDING,NonResidential,Office,Office,182540,DOWNTOWN,1972,14,1,186768,23775,162993,95,9.641e+06,218,47.607,-122.33
20395,HOLYOKE BUILDING,NonResidential,Office,Office,37120,DOWNTOWN,1900,5,1,39960,0,39960,77,2.1787e+06,48.2,47.605,-122.34
25217,SCHMITZ PARK ELEM SCHOOL (SPS-DISTRICT),School,K-12 School,K-12 School,51410,SOUTHWEST,1962,1,1,39199,0,39199,87,1.9689e+06,63.7,47.573,-122.4
814,WEST LAKE UNION CENTER,NonResidential,Office,Office,220416,MAGNOLIA / QUEEN ANNE,1994,10,1,381511,168978,212533,88,1.141e+07,93.4,47.633,-122.34
21218,HLF Vehicle Maintenance (A),NonResidential,Other - Services,Other - Services,16243,NORTHWEST,1958,2,1,26994,0,26994,,3.0768e+06,121.5,47.721,-122.34
23120,"WASTE MANAGEMENT MAINTENANCE SHED, BLDG A",NonResidential,Other - Services,Other - Services,18262,DELRIDGE NEIGHBORHOODS,1971,1,1,26232,0,26232,,3.0941e+06,121.4,47.53,-122.34
21319,IMPERIAL HOTEL BUILDING,NonResidential,Office,Office,35840,DOWNTOWN,1907,5,1,35840,0,35840,83,1.5388e+06,47.4,47.61,-122.34
24604,WEST SEATTLE VENTURE LLC,NonResidential,Mixed Use Property,Office,14452,SOUTHWEST,2004,5,1,35176,0,35176,,1.012e+06,5.9,47.573,-122.37
322,KIRO-TV,NonResidential,Office,Office,100734,DOWNTOWN,1968,3,1,100734,26731,74003,20,9.7887e+06,56.8,47.618,-122.35
516,AURORA SHOPPING CENTER,NonResidential,Strip Mall,Strip Mall,124231,NORTHWEST,1964,1,6,104919,0,104919,,9.8148e+06,218.6,47.725,-122.35
544,SEATTLE DISTRIBUTION CENTER BLDG B,NonResidential,Distribution Center,Distribution Center,49918,GREATER DUWAMISH,1967,1,1,51040,0,51040,72,1.5295e+06,41.6,47.542,-122.33
787,Seattle Lighting Distribution Center,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,101752,GREATER DUWAMISH,1952,2,1,115668,0,115668,,3.5347e+06,109.1,47.576,-122.34
625,2200 1ST AVE S,NonResidential,Office,Office,104869,GREATER DUWAMISH,1900,4,1,129000,25800,103200,80,4.3746e+06,25.4,47.584,-122.33
37008,CENTURYLINK EVENT CENTER & GARAGE,NonResidential,Parking,Parking,432903,GREATER DUWAMISH,1999,6,1,872409,538711,333698,,1.373e+07,283.9,47.593,-122.33
20984,BUILDING R,NonResidential,Office,Office,27185,GREATER DUWAMISH,1970,2,1,28304,0,28304,67,1.2694e+06,7.4,47.552,-122.33
437,VIRGINIA MASON- Lindeman Pavilion,Hospital,Medical Office,Medical Office,162000,EAST,1945,3,1,266000,104000,162000,65,1.3912e+07,188.9,47.61,-122.33
50062,STAYBRIDGE SUITES SEATTLE-FREMONT,NonResidential,Hotel,Hotel,88157,LAKE UNION,2014,4,1,126823,41539,85284,72,6.0424e+06,126,47.655,-122.35
24461,OUTDOOR EMPORIUM,NonResidential,Retail Store,Retail Store,34038,GREATER DUWAMISH,1975,1,1,43794,0,43794,88,1.1645e+06,22.5,47.588,-122.33
26119,AUTO-ROW BUILDING/PRICE-RAGEN.COM,NonResidential,Retail Store,Retail Store,19240,EAST,1910,2,1,24750,0,24750,,7.8234e+05,27.7,47.614,-122.32
49862,SCT TECHNICAL PAVILION - SEATTLE CENTER,NonResidential,Other,Other,29112,MAGNOLIA / QUEEN ANNE,1962,1,1,29000,0,29000,,2.4292e+06,53,47.621,-122.35
387,PUGET SOUND PLAZA,NonResidential,Office,Office,286538,DOWNTOWN,1960,21,1,552176,253750,298426,79,2.7077e+07,1116.7,47.609,-122.34
22801,MEADOWBROOK COMMUNITY CENTER POOL,NonResidential,Other - Recreation,Other - Recreation,31991,NORTH,1996,2,1,29258,0,29258,,8.0869e+06,305.5,47.705,-122.29
703,ELLIOTT WEST BLDG 3 - CELL THEREPEUTICS/ADMIN. OFF,NonResidential,Office,Office,104667,MAGNOLIA / QUEEN ANNE,2000,4,1,165111,55000,110111,1,2.5478e+07,147.8,47.623,-122.36
339,BRODERICK BUILDING,NonResidential,Office,Office,64712,DOWNTOWN,1900,7,1,89550,0,89550,85,3.9033e+06,48.8,47.603,-122.33
27562,AUTO-CHLOR SYSTEM,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,42844,GREATER DUWAMISH,1974,1,1,43380,0,43380,89,4.0506e+05,2.3,47.564,-122.32
661,NORDSTROM ELMER J MEDICAL TOWER CONDOMINIUM,NonResidential,Medical Office,Medical Office,197408,EAST,1985,10,1,422503,206580,215923,46,1.7255e+07,174.8,47.61,-122.32
23265,TERMINAL 102 PORT OF SEATTLE - CAMPUS,NonResidential,Other,Other,139557,GREATER DUWAMISH,1989,2,3,139557,0,139557,,7.8377e+06,173.2,47.57,-122.35
23762,WESTSIDE SCHOOL,School,Worship Facility,Worship Facility,35844,SOUTHWEST,1977,1,1,35844,0,35844,14,8.6613e+05,5,47.51,-122.37
22332,BALLARD SQUARE,NonResidential,Strip Mall,Strip Mall,25828,BALLARD,1928,2,1,30000,0,30000,,1.0949e+06,12,47.669,-122.39
9,WEST PRECINCT (SEATTLE POLICE),NonResidential,Police Station,Police Station,88830,DOWNTOWN,1999,2,1,97288,37198,60090,,1.2087e+07,292.7,47.616,-122.34
24991,CENTURYLINK: DUWAMISH MAIN,NonResidential,Other,Other,34733,GREATER DUWAMISH,1962,3,1,36140,0,36140,,4.6535e+06,27,47.541,-122.32
693,QUEEN ANNE SQUARE EAST,NonResidential,Office,Office,98668,MAGNOLIA / QUEEN ANNE,1982,5,1,98668,0,98668,90,4.3913e+06,25.5,47.625,-122.36
30,CORNISH (Main Campus Center),University,College/University,College/University,125000,DOWNTOWN,1928,7,1,126593,0,126593,,1.1344e+07,381.9,47.618,-122.34
21468,SHOWBOX BUILDING,NonResidential,Other - Entertainment/Public Assembly,Other - Entertainment/Public Assembly,11888,DOWNTOWN,1916,2,1,25920,0,25920,,1.2578e+06,7.3,47.608,-122.34
21508,WHITE & HITCHCOCK BUILDING/FLYING FISH,NonResidential,Food Service,Food Service,22710,DOWNTOWN,1930,2,1,22710,0,22710,,2.8776e+05,1.7,47.613,-122.35
23071,UW NORTHWEST HOSPITAL & MEDICAL CENTER - CAMPUS,Hospital,Hospital (General Medical & Surgical),Hospital (General Medical & Surgical),483520,NORTHWEST,1966,1,8,668027,266833,401194,85,9.8961e+07,3221.7,47.714,-122.34
493,FIRST WEST BUILDING,NonResidential,Office,Office,69691,MAGNOLIA / QUEEN ANNE,1971,5,1,94278,16101,78177,75,3.4849e+06,20.2,47.62,-122.36
590,4727 BUILDING,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,104786,GREATER DUWAMISH,1969,2,1,104786,0,104786,93,1.0284e+06,23,47.561,-122.33
627,ROEBLING BLDG,NonResidential,Office,Office,39870,GREATER DUWAMISH,1904,4,1,58970,0,58970,,1.7764e+06,39.8,47.595,-122.33
500,FRED MEYER LAKE CITY,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,121087,NORTH,1995,2,1,308965,187878,121087,78,1.4513e+07,265,47.724,-122.29
711,PUBLIC STORAGE (STORE 8495),NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,95330,DOWNTOWN,1912,5,1,95330,0,95330,81,1.2622e+06,7.3,47.607,-122.34
26909,CHI OMEGA SORORITY,NonResidential,Residence Hall/Dormitory,Residence Hall/Dormitory,27487,NORTHEAST,1926,3,1,27487,0,27487,82,1.5945e+06,65.3,47.663,-122.31
25711,BUSH GARDEN-RESTAURANT& LOUNGE-INACTIVE,NonResidential,Restaurant,Restaurant,28800,DOWNTOWN,1913,3,1,28800,0,28800,,8.9992e+05,28.7,47.597,-122.32
659,RUSSELL INVESTMENTS CENTER,NonResidential,Office,Office,914832,DOWNTOWN,2005,42,1,1592914,297457,1295457,97,5.6162e+07,418.3,47.607,-122.34
22962,OFFICE BUILDING,NonResidential,Office,Office,76105,NORTHWEST,1987,3,1,76105,27307,48798,91,2.4724e+06,14.3,47.708,-122.33
418,POLL BUILDING,NonResidential,Mixed Use Property,Office,39960,DOWNTOWN,1910,5,1,79920,0,79920,,3.0095e+06,17.5,47.608,-122.34
24716,PACIFIC COMMERCIAL,NonResidential,Refrigerated Warehouse,Refrigerated Warehouse,15667,GREATER DUWAMISH,1937,2,1,24430,0,24430,,1.2328e+06,22.5,47.59,-122.34
533,PUBLIC STORAGE (STORE 8168),NonResidential,Self-Storage Facility,Self-Storage Facility,130293,MAGNOLIA / QUEEN ANNE,1988,4,1,130293,23400,106893,,4.6553e+05,2.7,47.647,-122.38
24178,LOFT WORKSHOPS,NonResidential,Mixed Use Property,Office,11729,NORTH,2000,4,1,30948,0,30948,,9.1975e+05,5.3,47.702,-122.3
21328,KINDRED HOSPITAL SEATTLE,Hospital,Other/Specialty Hospital,Other/Specialty Hospital,52284,EAST,1964,4,1,49243,0,49243,,8.4762e+06,92.4,47.612,-122.33
21768,Trinity West Seattle,NonResidential,Worship Facility,Worship Facility,18278,SOUTHWEST,1957,1,1,23562,0,23562,96,5.3174e+05,3.1,47.534,-122.38
429,TERMINAL SALES OFFICE BUILDING,NonResidential,Office,Office,93879,DOWNTOWN,1923,11,1,102846,0,102846,96,5.3884e+06,297.6,47.611,-122.34
36,JANE ADDAMS MIDDLE (SPS-DISTRICT),School,K-12 School,K-12 School,164229,NORTH,1949,2,1,160645,0,160645,75,6.7792e+06,267.4,47.71,-122.29
26379,GEORGETOWN  INN,NonResidential,Hotel,Hotel,22452,GREATER DUWAMISH,1992,3,1,22452,0,22452,85,1.2442e+06,41.2,47.548,-122.32
801,UNIVERSITY VILLAGE NORTH (PARKING GARAGE/RETAIL/STORAGE),NonResidential,Parking,Parking,232685,NORTHEAST,2002,6,1,361398,272900,88498,,8.9155e+06,110.8,47.665,-122.3
24356,WA STATE DEPT OF CORRECTIONS,NonResidential,Office,Office,47368,GREATER DUWAMISH,1979,2,1,47368,0,47368,99,1.0427e+06,6,47.589,-122.33
20614,HGA INVESTMENT LLC,NonResidential,Self-Storage Facility,Self-Storage Facility,21900,GREATER DUWAMISH,1952,1,1,21900,0,21900,,1.3336e+06,65.4,47.565,-122.34
20439,WEST SEATTLE FAMILY YMCA,NonResidential,Fitness Center/Health Club/Gym,Fitness Center/Health Club/Gym,50000,SOUTHWEST,1984,1,1,60270,12485,47785,,6.1183e+06,207,47.562,-122.38
22230,THE BALLARD MARKET,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,29214,BALLARD,1966,1,1,29214,0,29214,8,9.3362e+06,200.9,47.67,-122.37
22883,KELLER SUPPLY,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,25547,MAGNOLIA / QUEEN ANNE,1955,2,1,39497,0,39497,9,1.7001e+06,26.7,47.649,-122.38
261,BOEING SOUTH PARK CAMPUS REPORT,NonResidential,Office,Office,240984,GREATER DUWAMISH,1980,2,16,334368,31874,302494,,5.3467e+07,694.6,47.524,-122.31
20916,ST JOHN EGAN HALL,NonResidential,Office,Office,23100,NORTHWEST,1960,1,1,20678,0,20678,98,7.7033e+05,32.4,47.686,-122.36
22511,W COMMODORE (combined with 22510),NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,56884,MAGNOLIA / QUEEN ANNE,1960,1,2,56884,0,56884,71,1.3883e+06,36.5,47.662,-122.39
23268,HARBOR MARINA BLDGS E & F,NonResidential,Office,Office,40657,GREATER DUWAMISH,1997,2,1,60657,20000,40657,57,2.8268e+06,20.4,47.57,-122.35
24959,HANFORD CNTR,NonResidential,Mixed Use Property,Other,10471,GREATER DUWAMISH,1952,1,1,35310,0,35310,,1.3315e+06,43.9,47.575,-122.34
35392,QFC UPTOWN,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,45980,MAGNOLIA / QUEEN ANNE,2006,3,1,45980,0,45980,90,8.3053e+06,186.7,47.625,-122.35
21465,PLAYHOUSE - SEATTLE CENTER,NonResidential,Other,Other,36314,MAGNOLIA / QUEEN ANNE,1962,1,1,36314,0,36314,,1.8328e+06,43.8,47.624,-122.35
19461,UNIVERSITY UNITARIAN CHURCH,NonResidential,Worship Facility,Worship Facility,22220,NORTHEAST,1957,2,1,22220,0,22220,90,7.7836e+05,31.4,47.677,-122.29
690,ONE CONVENTION PLACE,NonResidential,Office,Office,366554,DOWNTOWN,2001,23,1,538933,84660,454273,90,1.6021e+07,92.9,47.611,-122.33
24132,820 SOUTH ADAMS BUILDING,NonResidential,Refrigerated Warehouse,Refrigerated Warehouse,25200,GREATER DUWAMISH,1957,1,1,25520,0,25520,100,1.6108e+06,58.2,47.567,-122.32
411,JOSHUA GREEN BUILDING,NonResidential,Convenience Store without Gas Station,Office,87391,DOWNTOWN,1910,10,1,109266,0,109266,,,59.4,47.61,-122.34
3,WESTIN HOTEL (Parent Building),NonResidential,Hotel,Hotel,756493,DOWNTOWN,1969,41,3,956110,196718,759392,44,7.2586e+07,2113.2,47.614,-122.34
24358,PCF1XX/RE-PC/BRINKS,NonResidential,Mixed Use Property,Non-Refrigerated Warehouse,18871,GREATER DUWAMISH,1931,1,1,46183,0,46183,,1.8905e+06,26,47.589,-122.33
140,B F DAY ELEMENTARY (SPS-DISTRICT),School,K-12 School,K-12 School,66588,LAKE UNION,1991,3,1,66588,0,66588,71,2.1043e+06,12.2,47.655,-122.35
26126,THE SUMMIT,NonResidential,Other - Entertainment/Public Assembly,Other - Entertainment/Public Assembly,46560,EAST,1954,2,1,46560,0,46560,,6.9056e+05,16.8,47.614,-122.33
190,WEBSTER ST BUILDING (CEDAR GROVE),NonResidential,Office,Office,30000,GREATER DUWAMISH,2007,3,1,64015,0,64015,21,3.2421e+06,46.4,47.536,-122.32
647,FERGUSON,NonResidential,Distribution Center,Distribution Center,72719,GREATER DUWAMISH,1953,1,1,72719,0,72719,28,2.5817e+06,40.3,47.567,-122.35
24219,SB WAREHOUSE,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,39600,GREATER DUWAMISH,1952,1,1,39600,0,39600,91,4.684e+05,8.8,47.572,-122.33
274,PUBLIC STORAGE (STORE 27020),NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,67236,LAKE UNION,1991,3,1,75397,8161,67236,91,5.4918e+05,9.1,47.651,-122.34
21723,FAIRVIEW CHURCH,NonResidential,Worship Facility,Worship Facility,45680,NORTH,1908,3,1,45680,0,45680,79,2.077e+06,86,47.686,-122.32
797,WEST CORE RIVER STREET,NonResidential,Distribution Center,Distribution Center,71718,GREATER DUWAMISH,1969,1,1,71718,0,71718,100,1.5175e+05,0.9,47.543,-122.33
22548,SELF STORAGE MINI WAREHOUSE,NonResidential,Self-Storage Facility,Self-Storage Facility,39952,MAGNOLIA / QUEEN ANNE,1915,3,1,39952,0,39952,,57133,0.3,47.644,-122.38
361,YESLER BUILDING - KING COUNTY,NonResidential,Office,Office,96582,DOWNTOWN,1909,7,1,114395,8296,106099,99,2.7466e+06,68.1,47.602,-122.33
26593,DEL MAR BUILDING (HOTEL DEL MAR BUILDING),NonResidential,Office,Office,18400,DOWNTOWN,1900,4,1,22840,0,22840,64,8.8139e+05,5.1,47.601,-122.33
649,INscape ARTS BUILDING,NonResidential,Office,Office,76624,GREATER DUWAMISH,1930,4,1,76624,0,76624,91,2.7736e+06,30.5,47.595,-122.33
317,SEATTLE AQUARIUM,NonResidential,Other - Entertainment/Public Assembly,Other - Entertainment/Public Assembly,69400,DOWNTOWN,1980,2,1,53365,0,53365,,1.4833e+07,289,47.607,-122.34
26821,MCDONALD (SPS-DISTRICT),School,K-12 School,K-12 School,51026,LAKE UNION,1925,3,1,51360,0,51360,89,1.4031e+06,29,47.668,-122.33
864,PUBLIC STORAGE (STORE 8458),NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,80360,NORTH,1987,4,1,80360,0,80360,96,6.8602e+05,15.7,47.709,-122.3
25694,CENTURYLINK: CAMPUS CO,NonResidential,Other,Other,27022,NORTHEAST,1962,2,1,28586,0,28586,,5.0782e+06,29.5,47.657,-122.32
130,MCCLURE MIDDLE (SPS-DISTRICT),School,K-12 School,K-12 School,93218,MAGNOLIA / QUEEN ANNE,1964,2,1,93218,0,93218,86,2.8874e+06,82.9,47.637,-122.36
775,COLUMBIA CENTER,NonResidential,Office,Office,1680937,DOWNTOWN,1985,76,1,1952220,319400,1632820,86,9.2938e+07,779.1,47.605,-122.33
63,SHERATON GRAND SEATTLE - (Parent Building),NonResidential,Hotel,Hotel,994212,DOWNTOWN,1982,34,2,1083197,156583,926614,63,8.0475e+07,2439.9,47.611,-122.33
27833,SOUND MENTAL HEALTH,NonResidential,Urgent Care/Clinic/Other Outpatient,Urgent Care/Clinic/Other Outpatient,38879,EAST,1975,2,5,38879,11328,27551,,3.3717e+06,25,47.617,-122.31
27,KING COUNTY CORRECTIONAL FACILITY,NonResidential,Other,Other,385274,DOWNTOWN,1985,19,1,385274,45959,339315,,4.786e+07,1674.8,47.604,-122.33
21582,WASH STATE CREDIT UNION,NonResidential,Retail Store,Retail Store,10751,LAKE UNION,1993,3,1,25194,3764,21430,,1.8787e+06,54.7,47.632,-122.33
175,GARFIELD HIGH (SPS-DISTRICT),School,K-12 School,K-12 School,254523,CENTRAL,2008,3,1,254523,0,254523,60,1.3568e+07,349.3,47.606,-122.3
24593,E-Z MINI STORAGE (CUBESMART),NonResidential,Self-Storage Facility,Self-Storage Facility,25821,GREATER DUWAMISH,1925,2,1,25821,0,25821,,2.097e+05,1.2,47.582,-122.33
325,WESTLAKE MALL RETAIL PORTION,NonResidential,Enclosed Mall,Enclosed Mall,111077,DOWNTOWN,1989,4,1,111077,0,111077,,1.5335e+07,311.9,47.612,-122.34
195,KINDRED HOSPITAL- demolition,Hospital,Other/Specialty Hospital,Other/Specialty Hospital,47012,NORTH,1964,3,1,54224,0,54224,,7.2652e+06,227.2,47.707,-122.32
426,PORT OF SEATTLE - WORLD TRADE CENTER-WEST,NonResidential,Office,Office,70090,DOWNTOWN,1999,4,1,87500,17500,70000,72,4.5499e+06,26.4,47.611,-122.35
639,GREEN ROOM,NonResidential,Office,Office,60384,GREATER DUWAMISH,1930,3,1,60592,0,60592,16,5.0494e+06,41.6,47.583,-122.33
23418,ENG SUEY SUN PLAZA- DEMOLISHED DUE TO FIRE,NonResidential,Office,Office,20695,DOWNTOWN,1989,2,1,23472,0,23472,,1.6253e+06,45.4,47.597,-122.32
685,818 STEWART,NonResidential,Office,Office,243998,DOWNTOWN,2008,14,1,362033,124506,237527,87,1.5927e+07,99.9,47.616,-122.34
779,ADMINISTRATION BUILDING - KING COUNTY,NonResidential,Office,Office,204993,DOWNTOWN,1971,9,1,224857,0,224857,79,1.5272e+07,393,47.603,-122.33
2,PARAMOUNT HOTEL,NonResidential,Hotel,Hotel,83880,DOWNTOWN,1996,11,1,103566,15064,88502,61,8.3879e+06,291.5,47.613,-122.33
20735,ST ANNE SCHOOL,School,K-12 School,K-12 School,33550,MAGNOLIA / QUEEN ANNE,1922,3,1,32691,0,32691,99,9.7199e+05,29.2,47.631,-122.36
701,STAPLES BUILDING,NonResidential,Retail Store,Retail Store,50660,MAGNOLIA / QUEEN ANNE,1978,1,1,50660,0,50660,51,2.6615e+06,58.6,47.633,-122.38
25664,Ballou Wright Building,NonResidential,Office,Office,29720,EAST,1917,3,1,30720,0,30720,,1.2887e+06,24.4,47.615,-122.32
23691,WESTWOOD CHRISTIAN COMMUNITY,NonResidential,Worship Facility,Worship Facility,23706,DELRIDGE NEIGHBORHOODS,1963,1,1,23706,0,23706,78,4.0077e+05,12.9,47.52,-122.35
27830,ELYSIAN BREWING CO & OFFICES,NonResidential,Office,Office,23422,EAST,1920,3,1,23422,0,23422,65,2.7019e+06,127.4,47.614,-122.32
27668,VIRGINIA MASON - BAILEY BOUSHAY HOUSE,Hospital,Other/Specialty Hospital,Other/Specialty Hospital,34074,CENTRAL,1991,3,1,34074,0,34074,,8.522e+06,281.6,47.623,-122.3
20073,GRIFFIN BUILDING,NonResidential,Financial Office,Financial Office,25876,DOWNTOWN,1927,4,1,32356,0,32356,81,1.8572e+06,77.1,47.614,-122.34
27598,ST. ANDREW KIM,NonResidential,Worship Facility,Worship Facility,24955,NORTHWEST,2002,1,1,24955,0,24955,47,6.898e+05,21.5,47.715,-122.33
22818,WAREHOUSE,NonResidential,Distribution Center,Distribution Center,20100,DELRIDGE NEIGHBORHOODS,1980,1,1,24100,0,24100,22,8.0987e+05,30.1,47.558,-122.35
40447,SPU OPERATIONS CONTROL CENTER (OCC) COMPLEX,NonResidential,Mixed Use Property,Office,41097,GREATER DUWAMISH,1966,1,5,90072,0,90072,,8.4752e+06,265.6,47.579,-122.32
22440,OFFICE MAX,NonResidential,Retail Store,Retail Store,23500,BALLARD,1996,1,1,23500,0,23500,53,1.138e+06,12.1,47.663,-122.37
385,SKINNER BLDG,NonResidential,Office,Office,166919,DOWNTOWN,1927,8,1,226061,0,226061,98,6.0989e+06,357.5,47.609,-122.33
25361,DECATUR ELEMENTARY (SPS-DISTRICT),School,K-12 School,K-12 School,45370,NORTHEAST,1961,1,1,43578,0,43578,83,1.4368e+06,23.3,47.685,-122.28
366,UNIVERSITY DISTRICT BUILDING,NonResidential,Office,Office,79555,NORTHEAST,1961,5,1,99005,17934,81071,70,4.7285e+06,115.5,47.661,-122.32
84,PROVIDENCE MT ST VINCENT,Hospital,Senior Living Community,Senior Living Community,296313,SOUTHWEST,1922,5,1,371257,74944,296313,54,4.2792e+07,1709.9,47.558,-122.38
23017,FIFTH AVENUE MEDICAL CENTER,NonResidential,Medical Office,Medical Office,28841,NORTH,1988,3,1,28841,0,28841,55,1.4592e+06,32.2,47.707,-122.32
793,SEATTLE DESIGN CENTER,NonResidential,Office,Office,141917,GREATER DUWAMISH,1973,2,1,156188,0,156188,67,6.8439e+06,53.2,47.551,-122.33
24448,ALFA ROMEO AUTO SHOWROOM,NonResidential,Mixed Use Property,Parking,18000,EAST,1913,1,1,36000,18000,18000,59,1.1622e+06,25.6,47.613,-122.32
24332,4TH & FOREST,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,19430,GREATER DUWAMISH,1970,1,1,23750,0,23750,,5.2297e+05,3,47.578,-122.33
671,JEFFERSON TOWER,NonResidential,Medical Office,Medical Office,113739,CENTRAL,1987,7,1,108618,0,108618,66,5.9591e+06,34.6,47.606,-122.31
375,THIRD AND BROAD BLDG,NonResidential,Office,Office,274002,DOWNTOWN,1982,6,1,396626,124216,272410,64,2.0361e+07,118.1,47.618,-122.35
748,PIER 70,NonResidential,Office,Office,90345,DOWNTOWN,1902,3,1,116430,0,116430,,1.203e+07,250.1,47.615,-122.36
369,SEATTLE VAULT SELF STORAGE,NonResidential,Self-Storage Facility,Self-Storage Facility,93660,DOWNTOWN,1964,5,1,93660,0,93660,,8.7322e+05,5.1,47.616,-122.33
20047,UNIVERSITY VILLAGE SHOPPING CENTER BLDG A,NonResidential,Retail Store,Retail Store,31905,NORTHEAST,1956,2,1,39265,0,39265,,5.2388e+06,41.2,47.662,-122.3
21138,SATURN BUILDING,NonResidential,Mixed Use Property,Office,40313,LAKE UNION,1947,1,1,46958,0,46958,95,1.8189e+06,28.6,47.651,-122.35
49687,SAFEWAY #1885,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,33997,MAGNOLIA / QUEEN ANNE,2003,4,1,93997,60000,33997,23,8.9267e+06,84.9,47.624,-122.36
25513,ST THERESE PARISH,NonResidential,Worship Facility,Worship Facility,21850,CENTRAL,1900,1,1,21850,0,21850,100,3.1779e+05,13,47.611,-122.29
19863,815 2ND AVE,NonResidential,Bank Branch,Social/Meeting Hall,21970,DOWNTOWN,1924,2,0,21970,0,21970,26,6.3537e+05,3.7,47.604,-122.33
552,SOUTH SEATTLE DISTRIBUTION CENTER- 3800,NonResidential,Distribution Center,Distribution Center,162425,GREATER DUWAMISH,1968,1,1,174227,0,174227,25,9.3572e+06,197.1,47.57,-122.33
49699,PUBLIC STORAGE AND TRUCK RENTAL,NonResidential,Non-Refrigerated Warehouse,Non-Refrigerated Warehouse,293707,SOUTHEAST,1988,2,8,293707,0,293707,99,1.5442e+06,13.8,47.511,-122.28
20052,UNIVERSITY VILLAGE SHOPPING CENTER WEST BLDG,NonResidential,Retail Store,Retail Store,59396,NORTHEAST,1997,1,1,59142,0,59142,,4.6442e+06,42.6,47.662,-122.3
24240,CENTURYLINK: WEST CO,NonResidential,Other,Other,31332,SOUTHWEST,1920,1,1,31332,0,31332,,3.5733e+06,20.7,47.564,-122.39
26705,GREYBAR BUILDING,NonResidential,Office,Office,22000,DOWNTOWN,1930,2,1,36630,0,36630,90,5.4194e+05,3.1,47.599,-122.33
23065,NORTH DISTRICT MULTI-SERVICE CENTER - KING COUNTY,NonResidential,Medical Office,Medical Office,44667,NORTHWEST,1979,2,0,34759,0,34759,81,1.6331e+06,9.5,47.706,-122.33
20273,BROOKLYN PLAZA DORM,NonResidential,Residence Hall/Dormitory,Residence Hall/Dormitory,22000,NORTHEAST,1993,3,1,28572,7327,21245,16,2.7179e+06,88,47.657,-122.31
49695,CITY PLACE V (Arizona),NonResidential,Office,Office,400536,LAKE UNION,2012,11,1,526423,190431,335992,97,1.7464e+07,143.7,47.62,-122.34
24895,PROMENADE SOUTH 1,NonResidential,Supermarket/Grocery Store,Supermarket/Grocery Store,28470,CENTRAL,1980,1,0,30630,0,30630,27,6.0585e+06,35.1,47.599,-122.3
24653,GREENWOOD SHOPPING CENTER,NonResidential,Retail Store,Retail Store,51000,NORTHWEST,1953,1,1,51000,0,51000,95,1.298e+06,12.8,47.691,-122.36
29551,625 UNION STATION,NonResidential,Office,Office,61665,DOWNTOWN,2000,11,1,68452,0,68452,79,3.7824e+06,21.9,47.597,-122.33
476,FHCRC - YALE BLDG,Hospital,Office,Office,131592,LAKE UNION,2001,6,1,226592,95000,131592,49,1.1963e+07,80.9,47.626,-122.33
19895,CENTENNIAL BUILDING,NonResidential,Office,Office,25920,DOWNTOWN,1925,2,1,44098,13660,30438,53,1.4419e+06,42.2,47.613,-122.34
491,SOUTH TOWER,NonResidential,Office,Office,66747,MAGNOLIA / QUEEN ANNE,1970,5,1,87178,20416,66762,86,3.4931e+06,20.3,47.622,-122.36
26956,Kerry Hall,University,College/University,College/University,26000,EAST,1921,3,1,31900,0,31900,,2.4661e+06,95.1,47.625,-122.32
435,SEATTLE CONVENTION CENTER,NonResidential,Convention Center,Convention Center,1072000,DOWNTOWN,1990,6,1,1842925,686750,1156175,,7.386e+07,1832.2,47.612,-122.33
"""

In [ ]:
URL = "https://data.seattle.gov/resource/teqw-tu6e.csv?datayear=2016&$limit=10000"
CACHE = DATA_DIR / "seattle_2016.csv"
COLONNES = ["osebuildingid", "buildingname", "buildingtype", "epapropertytype", "largestpropertyusetype",
            "largestpropertyusetypegfa", "neighborhood", "yearbuilt", "numberoffloors", "numberofbuildings",
            "propertygfatotal", "propertygfaparking", "propertygfabuildings", "energystarscore",
            "siteenergyuse_kbtu", "totalghgemissions", "latitude", "longitude"]

if CACHE.exists():
    brut = pd.read_csv(CACHE)
    SOURCE = "cache local"
else:
    try:
        brut = pd.read_csv(URL)[COLONNES]
        brut.to_csv(CACHE, index=False)
        SOURCE = "Seattle Open Data (API Socrata)"
    except Exception as erreur:
        print("Téléchargement impossible :", erreur)
        brut = pd.read_csv(io.StringIO(DONNEES_SECOURS))
        SOURCE = "échantillon de secours (300 bâtiments)"
print(f"{len(brut)} bâtiments chargés depuis : {SOURCE}")

In [ ]:
print("Dimensions :", brut.shape)
print(brut.dtypes.to_string(), "\n")
manquants = brut.isna().mean().sort_values(ascending=False)
print("Part de valeurs manquantes (top 6) :")
print((manquants.head(6) * 100).round(1).astype(str) + " %")
print("\nFamilles de bâtiments :", brut["buildingtype"].value_counts().to_dict())
brut.head(3)

## 3. Nettoyage et feature engineering

Chaque correction est notée dans un **journal** (liste de dictionnaires) : c'est ce tableau qu'on montre en soutenance pour justifier « pourquoi il reste N bâtiments ». Règle du jeu : on ne supprime une ligne que si elle est **physiquement impossible** ou hors du périmètre.

In [ ]:
JOURNAL = []

def noter(action, avant, apres):
    JOURNAL.append({"étape": len(JOURNAL) + 1, "action": action, "lignes avant": avant, "lignes après": apres,
                    "supprimées": avant - apres})
    print(f"{action:70s} {avant:5d} → {apres:5d}")

df = brut.copy()
n = len(df)
df = df[~df["buildingtype"].str.startswith("Multifamily")]          # périmètre : non résidentiel
noter("Périmètre : retrait des immeubles résidentiels (Multifamily)", n, len(df))

Ensuite les valeurs impossibles, puis les **outliers** : quelques valeurs de consommation sont aberrantes (des centaines de fois la médiane pour une surface banale). Plutôt que de couper sur la consommation brute (qui dépend de la taille), on regarde l'**intensité énergétique** (kBtu par pied carré) et on coupe les 1 % extrêmes de chaque côté.

In [ ]:
n = len(df)
df = df.dropna(subset=["siteenergyuse_kbtu", "totalghgemissions"])
noter("Cibles manquantes", n, len(df))

n = len(df)
df = df[(df["siteenergyuse_kbtu"] > 0) & (df["totalghgemissions"] > 0)]
noter("Cibles nulles ou négatives (bâtiment vide ou erreur de saisie)", n, len(df))

n = len(df)
df = df[(df["propertygfatotal"] > 0) & (df["propertygfaparking"] < df["propertygfatotal"])]
noter("Surface nulle ou parking ≥ surface totale", n, len(df))

intensite = df["siteenergyuse_kbtu"] / df["propertygfatotal"]
bas, haut = intensite.quantile([0.01, 0.99])
n = len(df)
df = df[(intensite >= bas) & (intensite <= haut)]
noter(f"Intensité énergétique hors [{bas:.1f} ; {haut:.0f}] kBtu/sq ft (1 % de chaque côté)", n, len(df))

n = len(df)
df = df[df["yearbuilt"].between(1850, 2016)]
noter("Année de construction impossible", n, len(df))
df = df.reset_index(drop=True)

### Nouvelles variables

- `age` : 2016 − année de construction (plus parlant qu'une année brute).
- `surface_hors_parking` : la surface totale moins le parking — un parking consomme peu, c'est le bâtiment qui chauffe.
- `part_parking` : part de parking dans la surface totale.
- `part_usage1` : part de la surface occupée par l'usage principal (un immeuble « 100 % bureaux » ne ressemble pas à un immeuble mixte).
- `log_conso` et `log_ghg` : les **cibles en log** (`np.log1p`), pour ramener des ordres de grandeur très différents sur une échelle où l'erreur est relative.

In [ ]:
df["age"] = 2016 - df["yearbuilt"]
df["surface_hors_parking"] = (df["propertygfatotal"] - df["propertygfaparking"]).clip(lower=1)
df["part_parking"] = (df["propertygfaparking"] / df["propertygfatotal"]).clip(0, 1)
df["part_usage1"] = (df["largestpropertyusetypegfa"] / df["propertygfatotal"]).clip(0, 1).fillna(1.0)
df["log_conso"] = np.log1p(df["siteenergyuse_kbtu"])
df["log_ghg"] = np.log1p(df["totalghgemissions"])
df["log_surface"] = np.log1p(df["surface_hors_parking"])
print(df[["age", "surface_hors_parking", "part_parking", "part_usage1", "log_conso", "log_ghg"]].describe().round(2))

**À toi (1)** · Crée `surface_par_etage` = surface hors parking divisée par le nombre d'étages. Attention : certains bâtiments ont `numberoffloors = 0` (donnée manquante déguisée) → compte au moins 1 étage.

<details><summary>Indice</summary>

`df["numberoffloors"].clip(lower=1)` remplace les 0 par 1. Vérifie ensuite qu'il n'y a ni `inf` ni `NaN`.
</details>

In [ ]:
# À toi : remplace le calcul provisoire par le bon
df["surface_par_etage"] = df["surface_hors_parking"]      # provisoire : à diviser par le nombre d'étages (≥ 1)
print(df["surface_par_etage"].describe().round(0))

In [ ]:
verifier("Exercice 1 · surface_par_etage sans NaN ni inf", df["surface_par_etage"].notna().all() and np.isfinite(df["surface_par_etage"]).all())
verifier("Exercice 1 · surface_par_etage plus petite que la surface totale",
         (df["surface_par_etage"] <= df["surface_hors_parking"]).all() and df["surface_par_etage"].median() < df["surface_hors_parking"].median())

<details><summary>Solution</summary>

```python
df["surface_par_etage"] = df["surface_hors_parking"] / df["numberoffloors"].clip(lower=1)
print(df["surface_par_etage"].describe().round(0))
```
</details>

**À toi (2)** · L'usage principal (`largestpropertyusetype`) compte des dizaines de modalités dont certaines n'ont que 2 ou 3 bâtiments : impossible d'apprendre quelque chose dessus. Crée `usage_principal` : l'usage s'il apparaît **au moins 20 fois**, sinon `"Autre"`. Les valeurs manquantes deviennent aussi `"Autre"`.

<details><summary>Indice</summary>

`comptes = df["largestpropertyusetype"].value_counts()` puis `frequents = comptes[comptes >= 20].index` et `.where(...isin(frequents), "Autre")`.
</details>

In [ ]:
# À toi : regroupe les usages rares dans "Autre"
df["usage_principal"] = df["largestpropertyusetype"].fillna("Autre")     # provisoire : rien n'est regroupé
print(df["usage_principal"].value_counts().head(12))
print("Nombre d'usages distincts :", df["usage_principal"].nunique())

In [ ]:
comptes_usage = df["usage_principal"].value_counts()
verifier("Exercice 2 · « Autre » existe et regroupe les usages rares", "Autre" in comptes_usage.index and comptes_usage.drop("Autre").min() >= 20)
verifier("Exercice 2 · aucune valeur manquante", df["usage_principal"].notna().all())

<details><summary>Solution</summary>

```python
comptes = df["largestpropertyusetype"].value_counts()
frequents = comptes[comptes >= 20].index
df["usage_principal"] = df["largestpropertyusetype"].where(df["largestpropertyusetype"].isin(frequents), "Autre").fillna("Autre")
print(df["usage_principal"].value_counts().head(12))
print("Nombre d'usages distincts :", df["usage_principal"].nunique())
```
</details>

In [ ]:
df["neighborhood"] = df["neighborhood"].str.upper().fillna("INCONNU")   # DOWNTOWN et Downtown = même quartier
journal = pd.DataFrame(JOURNAL)
print(f"Bâtiments conservés : {len(df)} sur {len(brut)} ({len(df) / len(brut):.0%})")
journal

## 4. Analyse exploratoire

Cinq indicateurs, un graphique chacun, une phrase de lecture. L'objectif n'est pas de tout décrire mais de comprendre **ce qui fait varier la consommation** — et donc quel type de modèle a une chance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
df["siteenergyuse_kbtu"].plot.hist(bins=60, ax=axes[0], color="tab:orange")
axes[0].set_title("Consommation brute (kBtu) : très asymétrique")
df["log_conso"].plot.hist(bins=40, ax=axes[1], color="tab:blue")
axes[1].set_title("log(1 + consommation) : presque une cloche")
axes[2].scatter(df["log_surface"], df["log_conso"], s=6, alpha=0.5)
axes[2].set_xlabel("log(surface hors parking)"); axes[2].set_ylabel("log(consommation)"); axes[2].set_title("Surface ↔ consommation")
plt.tight_layout(); plt.show()
print(f"Médiane : {df['siteenergyuse_kbtu'].median() / 1e6:.2f} MkBtu · max : {df['siteenergyuse_kbtu'].max() / 1e6:.0f} MkBtu")

**Lecture** : sans le log, une poignée de très gros bâtiments (hôpitaux, campus) écraserait tout ; en log, la cible est symétrique et une erreur de 0,3 signifie « à 35 % près » quel que soit le bâtiment. À droite, la surface est de loin la relation la plus forte — quasi linéaire en log-log.

**À toi (3)** · Mesure la corrélation de Pearson entre `log_surface` et `log_conso`, et range-la dans `r_surface`. Puis calcule la corrélation entre `age` et `log_conso` dans `r_age`. Laquelle est la plus utile pour prédire ?

<details><summary>Indice</summary>

`df["a"].corr(df["b"])` renvoie directement le coefficient de Pearson.
</details>

In [ ]:
# À toi
r_surface = None
r_age = None
print("r(surface, conso) =", r_surface, "   r(âge, conso) =", r_age)

In [ ]:
verifier("Exercice 3 · r_surface est une corrélation forte et positive", r_surface is not None and 0.5 < r_surface < 1)
verifier("Exercice 3 · r_age est bien plus faible en valeur absolue", r_age is not None and abs(r_age) < abs(r_surface))

<details><summary>Solution</summary>

```python
r_surface = df["log_surface"].corr(df["log_conso"])
r_age = df["age"].corr(df["log_conso"])
print("r(surface, conso) =", round(r_surface, 3), "   r(âge, conso) =", round(r_age, 3))
```
</details>

In [ ]:
df["intensite"] = df["siteenergyuse_kbtu"] / df["propertygfatotal"]          # kBtu par sq ft : compare des bâtiments de tailles différentes
usages_nommes = df.loc[~df["usage_principal"].isin(["Autre", "Other"]), "usage_principal"]      # on écarte les fourre-tout
top_usages = usages_nommes.value_counts().head(10).index
par_usage = df[df["usage_principal"].isin(top_usages)].groupby("usage_principal")["intensite"].median().sort_values()
par_quartier = df.groupby("neighborhood")["intensite"].agg(["median", "count"]).query("count >= 15").sort_values("median")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
par_usage.plot.barh(color="tab:green", ax=axes[0]); axes[0].set_title("Intensité médiane par usage principal (kBtu / sq ft)")
par_quartier["median"].plot.barh(color="tab:purple", ax=axes[1]); axes[1].set_title("Par quartier (≥ 15 bâtiments)")
plt.tight_layout(); plt.show()

**Lecture** : à surface égale, l'usage principal multiplie l'intensité par 3 à 5 entre un entrepôt et un supermarché ou un hôpital — 2e variable clé, **catégorielle** : il faudra l'encoder. Les écarts entre quartiers sont bien plus faibles : le quartier reflète surtout le mélange d'usages (le centre-ville concentre bureaux et hôtels). Variable secondaire.

In [ ]:
df["decennie"] = (df["yearbuilt"] // 10 * 10).clip(lower=1900)
variables_num = ["log_conso", "log_ghg", "log_surface", "surface_par_etage", "age", "numberoffloors", "part_parking", "part_usage1", "energystarscore"]
corr = df[variables_num].corr()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), width_ratios=[1, 1.3])
df.groupby("decennie")["intensite"].median().plot(marker="o", ax=axes[0])
axes[0].set_ylabel("intensité médiane (kBtu / sq ft)"); axes[0].set_title("Par décennie de construction"); axes[0].grid(alpha=0.3)
im = axes[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr))); axes[1].set_xticklabels(corr.columns, rotation=60, ha="right"); axes[1].set_yticks(range(len(corr))); axes[1].set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[1].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=axes[1]); axes[1].set_title("Corrélations"); plt.tight_layout(); plt.show()

**Lecture** : pas de tendance nette avec l'âge — les bâtiments récents ne consomment pas moins par pied carré (plus climatisés, plus équipés). L'âge servira surtout en **interaction** avec l'usage, ce que les arbres savent capturer. Sur la matrice : surface ≫ étages > tout le reste ; l'ENERGY STAR score est négativement corrélé à la consommation (logique) mais rappelle-toi qu'il manque pour un tiers des bâtiments.

**Ce qui oriente le choix du modèle** : une relation quasi linéaire en log avec la surface (un modèle linéaire fera déjà quelque chose), une variable catégorielle très informative (usage), des effets non linéaires et des interactions (usage × âge, étages) → les **modèles à base d'arbres** (forêt, gradient boosting) sont les candidats naturels. Les deux cibles sont corrélées à 0,9 : on travaille d'abord sur la consommation, puis on vérifie que tout se transpose aux émissions.

In [ ]:
print("=== Rapport · section 4 (indicateurs) ===")
print(f"Bâtiments analysés : {len(df)}   |   médiane conso : {df['siteenergyuse_kbtu'].median() / 1e6:.2f} MkBtu   |   médiane intensité : {df['intensite'].median():.0f} kBtu/sq ft")
print(f"Corrélation log surface ↔ log conso : {df['log_surface'].corr(df['log_conso']):.2f}   |   ENERGY STAR manquant : {df['energystarscore'].isna().mean():.0%}")
print(f"Usage le plus énergivore (intensité médiane) : {par_usage.index[-1]} ({par_usage.iloc[-1]:.0f})   |   le plus sobre : {par_usage.index[0]} ({par_usage.iloc[0]:.0f})")

## 5. Modèles candidats

**Protocole** : 80 % apprentissage / 20 % test (jamais touché avant la fin), et **validation croisée à 5 plis** sur l'apprentissage pour comparer les modèles. Chaque modèle est un `Pipeline` sklearn : imputation + standardisation des variables numériques, encodage one-hot des catégories (avec regroupement automatique des modalités rares), puis le modèle. On reporte le RMSE en log (métrique principale), le MAE en unités réelles et le R².

On commence par une **baseline « bête »** : prédire la médiane pour tout le monde. Tout modèle qui ne fait pas nettement mieux ne vaut pas son coût.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

VARS_NUM = ["log_surface", "surface_par_etage", "age", "numberoffloors", "numberofbuildings", "part_parking", "part_usage1", "latitude", "longitude"]
VARS_CAT = ["usage_principal", "buildingtype", "neighborhood"]
CIBLE = "log_conso"

X = df[VARS_NUM + VARS_CAT]
y = df[CIBLE]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
CV = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Apprentissage : {len(X_train)} bâtiments · test : {len(X_test)}")

In [ ]:
def faire_preprocesseur(brutes=()):
    """Imputation + standardisation des numériques, one-hot des catégories ; `brutes` = colonnes passées telles quelles (NaN compris)."""
    num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False))])
    return ColumnTransformer([("num", num, VARS_NUM), ("brut", "passthrough", list(brutes)), ("cat", cat, VARS_CAT)])

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

TABLEAU = []

def evaluer(nom, modele, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test, cv=True, echelle=1e6, unite="MkBtu"):
    """CV 5 plis sur l'apprentissage + mesure sur le test, en log et en unités réelles. Renvoie le dict (et l'ajoute au tableau)."""
    debut = time.time()
    pipe = Pipeline([("prep", faire_preprocesseur()), ("modele", modele)])
    rmse_cv = -cross_val_score(pipe, X_train, y_train, cv=CV, scoring="neg_root_mean_squared_error").mean() if cv else np.nan
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    reel_vrai, reel_pred = np.expm1(y_test), np.expm1(pred)
    ligne = {"modèle": nom, "RMSE CV (log)": rmse_cv, "RMSE test (log)": rmse(y_test, pred), "MAE test (log)": mean_absolute_error(y_test, pred),
             "R² test (log)": r2_score(y_test, pred), f"MAE test ({unite})": mean_absolute_error(reel_vrai, reel_pred) / echelle,
             "R² test (réel)": r2_score(reel_vrai, reel_pred), "temps (s)": time.time() - debut}
    TABLEAU.append(ligne)
    print(f"{nom:28s} RMSE CV {rmse_cv:.3f} · RMSE test {ligne['RMSE test (log)']:.3f} · MAE {ligne[f'MAE test ({unite})']:.2f} {unite} · R² {ligne['R² test (log)']:.2f} · {ligne['temps (s)']:.1f}s")
    return pipe

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

modele_baseline = evaluer("Baseline (médiane)", DummyRegressor(strategy="median"))
modele_ridge = evaluer("Ridge (linéaire)", Ridge(alpha=1.0))
N_ARBRES = 200 if MODE_RAPIDE else 500
modele_rf = evaluer("Random Forest", RandomForestRegressor(n_estimators=N_ARBRES, min_samples_leaf=2, random_state=SEED, n_jobs=-1))
modele_xgb = evaluer("XGBoost (défaut)", XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1))

In [ ]:
tableau = pd.DataFrame(TABLEAU).set_index("modèle").round(3)
ax = tableau[["RMSE CV (log)", "RMSE test (log)"]].plot.barh(figsize=(8, 3.5))
ax.set_xlabel("RMSE en log (plus petit = mieux)"); ax.invert_yaxis(); ax.set_title("Comparaison des candidats")
plt.tight_layout(); plt.show()
tableau

**Lecture** : les trois modèles divisent l'erreur de la baseline par deux ; les arbres sont légèrement devant en log. Regarde la colonne **R² test (réel)** : Ridge y fait mieux, car la relation surface → consommation est quasi linéaire en log et un modèle linéaire extrapole mieux sur les quelques géants qui pèsent lourd en unités réelles. Le choix de la métrique n'est jamais neutre.

**À toi (4)** · La 2e cible : les émissions `log_ghg`. Réutilise `evaluer()` avec le même XGBoost par défaut, mais sur `y_ghg_train` / `y_ghg_test` (mêmes lignes que le split précédent). Range le pipeline dans `modele_ghg` : le résultat est-il aussi bon que pour la consommation ?

<details><summary>Indice</summary>

`y_ghg_train = df.loc[X_train.index, "log_ghg"]` — puis `evaluer("XGBoost → GHG", XGBRegressor(...), y_train=y_ghg_train, y_test=y_ghg_test, echelle=1, unite="t CO₂e")` (les émissions sont en tonnes, pas en kBtu).
</details>

In [ ]:
# À toi
y_ghg_train = df.loc[X_train.index, "log_ghg"]
y_ghg_test = df.loc[X_test.index, "log_ghg"]
modele_ghg = None

In [ ]:
verifier("Exercice 4 · modele_ghg est un pipeline entraîné", modele_ghg is not None and hasattr(modele_ghg, "predict"))
verifier("Exercice 4 · le R² sur les émissions dépasse 0,3", lambda: r2_score(y_ghg_test, modele_ghg.predict(X_test)) > 0.3)
if modele_ghg is not None:
    TABLEAU[:] = [l for l in TABLEAU if l["modèle"] != "XGBoost → GHG"]      # on garde le tableau principal sur la conso

<details><summary>Solution</summary>

```python
y_ghg_train = df.loc[X_train.index, "log_ghg"]
y_ghg_test = df.loc[X_test.index, "log_ghg"]
modele_ghg = evaluer("XGBoost → GHG", XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1),
                     y_train=y_ghg_train, y_test=y_ghg_test, echelle=1, unite="t CO₂e")
```
</details>

In [ ]:
print("=== Rapport · section 5 (modèles candidats) ===")
print(tableau[["RMSE CV (log)", "RMSE test (log)", "MAE test (MkBtu)", "R² test (log)", "temps (s)"]].to_string())
meilleur = tableau["RMSE CV (log)"].idxmin()
print(f"\nMeilleur candidat en CV : {meilleur} · gain vs baseline : {1 - tableau.loc[meilleur, 'RMSE CV (log)'] / tableau.loc['Baseline (médiane)', 'RMSE CV (log)']:.0%} de RMSE en moins")

## 6. Tuning : XGBoost + `RandomizedSearchCV`

XGBoost a une dizaine d'hyperparamètres qui interagissent (profondeur, taux d'apprentissage, nombre d'arbres, sous-échantillonnage…). Une grille exhaustive coûterait des milliers d'entraînements ; `RandomizedSearchCV` tire **au hasard** `n_iter` combinaisons dans des distributions et garde la meilleure en validation croisée — à budget égal, c'est presque toujours plus efficace qu'une grille ([Bergstra & Bengio, 2012](https://www.jmlr.org/papers/v13/bergstra12a.html)).

En `MODE_RAPIDE`, 8 tirages × 3 plis ; en mode complet, 40 tirages × 5 plis (≈ 3-4 min sur Colab).

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

ESPACE = {
    "modele__n_estimators": randint(200, 800),
    "modele__max_depth": randint(3, 9),
    "modele__learning_rate": loguniform(0.01, 0.2),
    "modele__subsample": uniform(0.6, 0.4),            # tiré dans [0,6 ; 1,0]
    "modele__colsample_bytree": uniform(0.5, 0.5),
    "modele__min_child_weight": randint(1, 10),
    "modele__reg_lambda": loguniform(0.1, 10),
}
N_ITER, N_PLIS = (8, 3) if MODE_RAPIDE else (40, 5)
pipe_xgb = Pipeline([("prep", faire_preprocesseur()), ("modele", XGBRegressor(random_state=SEED, n_jobs=1))])
recherche = RandomizedSearchCV(pipe_xgb, ESPACE, n_iter=N_ITER, cv=KFold(N_PLIS, shuffle=True, random_state=SEED),
                               scoring="neg_root_mean_squared_error", random_state=SEED, n_jobs=-1, verbose=0)
debut = time.time()
recherche.fit(X_train, y_train)
print(f"{N_ITER} combinaisons × {N_PLIS} plis en {time.time() - debut:.0f} s")
print("Meilleur RMSE CV (log) :", round(-recherche.best_score_, 3))
MEILLEURS_PARAMS = {k.replace("modele__", ""): (round(float(v), 3) if isinstance(v, float) else int(v)) for k, v in recherche.best_params_.items()}
print("Meilleurs paramètres :", MEILLEURS_PARAMS)

In [ ]:
modele_xgb_tune = evaluer("XGBoost (réglé)", XGBRegressor(**MEILLEURS_PARAMS, random_state=SEED, n_jobs=-1))
tableau = pd.DataFrame(TABLEAU).set_index("modèle").round(3)
display(tableau)
pred_test = modele_xgb_tune.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(y_test, pred_test, s=10, alpha=0.6); axes[0].plot([10, 20], [10, 20], "r--")
axes[0].set_xlabel("réel (log)"); axes[0].set_ylabel("prédit (log)"); axes[0].set_title("Prédit vs réel (test, modèle réglé)")
residus = y_test - pred_test
axes[1].hist(residus, bins=40); axes[1].set_title(f"Résidus (log) · écart-type {residus.std():.2f}")
plt.tight_layout(); plt.show()

**Courbes avant / après** : on suit le RMSE à mesure que les arbres s'ajoutent, pour le XGBoost par défaut et pour le XGBoost réglé (sur un découpage apprentissage / validation interne, le test reste intouché). Ce qu'il faut regarder : le modèle par défaut colle très vite aux données d'apprentissage (pointillés qui plongent) alors que sa validation stagne — sur-apprentissage ; le modèle réglé apprend plus lentement et garde un écart apprentissage / validation plus petit.

In [ ]:
X_app, X_val, y_app, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)
prep = faire_preprocesseur().fit(X_app)
X_app_p, X_val_p = prep.transform(X_app), prep.transform(X_val)

def courbe(params, nom):
    m = XGBRegressor(**params, random_state=SEED, n_jobs=-1)
    m.fit(X_app_p, y_app, eval_set=[(X_app_p, y_app), (X_val_p, y_val)], verbose=False)
    r = m.evals_result()
    plt.plot(r["validation_0"]["rmse"], "--", alpha=0.6, label=f"{nom} · apprentissage")
    plt.plot(r["validation_1"]["rmse"], label=f"{nom} · validation")

courbe({"n_estimators": 300, "learning_rate": 0.1, "max_depth": 6}, "défaut")
courbe(MEILLEURS_PARAMS, "réglé")
plt.ylim(0, 1.2); plt.xlabel("nombre d'arbres"); plt.ylabel("RMSE (log)"); plt.legend(fontsize=8); plt.title("Avant / après réglage")
plt.show()

**À toi (5)** · La recherche a exploré `subsample` et `colsample_bytree` ; XGBoost a aussi `gamma` (pénalité pour créer une feuille) et `reg_alpha` (régularisation L1). Construis `MON_ESPACE` = l'espace précédent + ces deux paramètres (`loguniform(0.01, 5)` pour `gamma`, `loguniform(0.01, 10)` pour `reg_alpha`), lance `ma_recherche` avec `n_iter=4` (rapide) et compare le meilleur score.

<details><summary>Indice</summary>

`MON_ESPACE = {**ESPACE, "modele__gamma": loguniform(0.01, 5), "modele__reg_alpha": loguniform(0.01, 10)}` puis le même `RandomizedSearchCV(...)`.
</details>

In [ ]:
# À toi
MON_ESPACE = dict(ESPACE)            # à compléter avec gamma et reg_alpha
ma_recherche = None                  # RandomizedSearchCV(...) puis .fit(X_train, y_train)

In [ ]:
verifier("Exercice 5 · MON_ESPACE contient gamma et reg_alpha", "modele__gamma" in MON_ESPACE and "modele__reg_alpha" in MON_ESPACE)
verifier("Exercice 5 · ma_recherche a été entraînée", ma_recherche is not None and hasattr(ma_recherche, "best_score_"))

<details><summary>Solution</summary>

```python
MON_ESPACE = {**ESPACE, "modele__gamma": loguniform(0.01, 5), "modele__reg_alpha": loguniform(0.01, 10)}
ma_recherche = RandomizedSearchCV(pipe_xgb, MON_ESPACE, n_iter=4 if MODE_RAPIDE else 20, cv=KFold(N_PLIS, shuffle=True, random_state=SEED),
                                  scoring="neg_root_mean_squared_error", random_state=SEED + 1, n_jobs=-1).fit(X_train, y_train)
print("RMSE CV avec gamma + reg_alpha :", round(-ma_recherche.best_score_, 3), " vs ", round(-recherche.best_score_, 3))
```
</details>

In [ ]:
print("=== Rapport · section 6 (tuning) ===")
avant, apres = tableau.loc["XGBoost (défaut)"], tableau.loc["XGBoost (réglé)"]
print(f"RMSE CV (log)  : {avant['RMSE CV (log)']:.3f} → {apres['RMSE CV (log)']:.3f}")
print(f"RMSE test (log): {avant['RMSE test (log)']:.3f} → {apres['RMSE test (log)']:.3f}   |   MAE test : {avant['MAE test (MkBtu)']:.2f} → {apres['MAE test (MkBtu)']:.2f} MkBtu")
print(f"R² test (log)  : {avant['R² test (log)']:.3f} → {apres['R² test (log)']:.3f}   |   {N_ITER} combinaisons × {N_PLIS} plis")
print("Paramètres retenus :", MEILLEURS_PARAMS)

## 7. Interprétation

Un RMSE ne convainc pas un décideur : il veut savoir **quelles variables** pilotent la prédiction et **pourquoi** tel bâtiment est prédit énergivore. Les **valeurs de Shapley** (librairie `shap`) répartissent chaque prédiction entre les variables, avec une garantie : la somme des contributions = prédiction − moyenne.

In [ ]:
import shap

prep_final = modele_xgb_tune.named_steps["prep"]
noms_variables = [n.split("__", 1)[1] for n in prep_final.get_feature_names_out()]
X_test_p = pd.DataFrame(prep_final.transform(X_test), columns=noms_variables, index=X_test.index)

explainer = shap.Explainer(modele_xgb_tune.named_steps["modele"], feature_names=noms_variables)
shap_values = explainer(X_test_p)
print("Valeurs SHAP calculées :", shap_values.values.shape, "(bâtiments × variables)")
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title("Importance globale : chaque point = un bâtiment ; couleur = valeur de la variable"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
shap.plots.scatter(shap_values[:, "log_surface"], ax=axes[0], show=False); axes[0].set_title("Dépendance : surface")
shap.plots.scatter(shap_values[:, "age"], ax=axes[1], show=False); axes[1].set_title("Dépendance : âge")
plt.tight_layout(); plt.show()

**Lecture** : la surface domine (logique), puis l'usage principal (les entrepôts tirent la prédiction vers le bas) et l'âge. Sur le beeswarm, une valeur élevée de `log_surface` (rouge) pousse la prédiction vers le haut. En dépendance, la contribution de la surface est quasi linéaire en log (le modèle a retrouvé la loi d'échelle) ; celle de l'âge est plate avec un léger creux pour les bâtiments très récents — confirmation de l'EDA.

### La question ENERGY STAR

Le score ENERGY STAR (1-100) résume la performance d'un bâtiment par rapport à ses semblables — mais il est manquant pour un tiers des bâtiments et coûte cher à établir. On compare **le même XGBoost réglé** avec et sans cette variable (XGBoost gère nativement les valeurs manquantes), sur le test complet et sur le sous-ensemble où le score est connu.

In [ ]:
VARS_AVEC = VARS_NUM + ["energystarscore"] + VARS_CAT
X_avec = df[VARS_AVEC]
Xa_train, Xa_test = X_avec.loc[X_train.index], X_avec.loc[X_test.index]

# le score passe en « brut » : ses NaN arrivent tels quels à XGBoost, qui sait les gérer
modele_avec = Pipeline([("prep", faire_preprocesseur(brutes=["energystarscore"])), ("modele", XGBRegressor(**MEILLEURS_PARAMS, random_state=SEED, n_jobs=-1))])
modele_avec.fit(Xa_train, y_train)
masque_connu = Xa_test["energystarscore"].notna()
comparaison = pd.DataFrame({
    "sans score": [rmse(y_test, modele_xgb_tune.predict(X_test)), rmse(y_test[masque_connu], modele_xgb_tune.predict(X_test[masque_connu]))],
    "avec score": [rmse(y_test, modele_avec.predict(Xa_test)), rmse(y_test[masque_connu], modele_avec.predict(Xa_test[masque_connu]))],
}, index=["RMSE test (log) · tous", f"RMSE test (log) · score connu ({masque_connu.sum()} bât.)"]).round(3)
sv_avec = shap.Explainer(modele_avec.named_steps["modele"])(modele_avec.named_steps["prep"].transform(Xa_test))
noms_avec = [n.split("__", 1)[1] for n in modele_avec.named_steps["prep"].get_feature_names_out()]
importance_avec = pd.Series(np.abs(sv_avec.values).mean(axis=0), index=noms_avec).sort_values(ascending=False)
rang_score = list(importance_avec.index).index("energystarscore") + 1
gain = 1 - comparaison.loc[comparaison.index[1], "avec score"] / comparaison.loc[comparaison.index[1], "sans score"]
print(f"ENERGY STAR score : {rang_score}e variable la plus importante (|SHAP| moyen {importance_avec['energystarscore']:.3f})")
print(f"Gain de RMSE là où le score est connu : {gain:.0%}")
comparaison

**Réponse** : quand le score est connu, il devient une des variables les plus importantes et réduit l'erreur de façon mesurable — il capte l'efficacité réelle (isolation, équipements) que la surface et l'usage ignorent. Mais pour un tiers des bâtiments il est absent, et le modèle sans score reste utilisable partout. Recommandation type P4 : **garder deux modèles** (avec / sans) ou, si la ville doit choisir, ne financer le score que pour les gros consommateurs où le gain absolu justifie le coût.

**À toi (6)** · Analyse d'erreurs : construis `pires`, un tableau des **5 bâtiments du test les plus mal prédits en unités réelles** (colonnes `buildingname`, `usage_principal`, `reel_MkBtu`, `predit_MkBtu`, `erreur_MkBtu`), trié par erreur absolue décroissante. Que remarques-tu sur leur type ?

<details><summary>Indice</summary>

`erreurs = df.loc[X_test.index, ["buildingname", "usage_principal"]].copy()` puis ajoute `reel_MkBtu = np.expm1(y_test) / 1e6`, `predit_MkBtu = np.expm1(pred_test) / 1e6`, la différence, et `.reindex(erreurs["erreur_MkBtu"].abs().sort_values(ascending=False).index).head(5)`.
</details>

In [ ]:
# À toi
erreurs = df.loc[X_test.index, ["buildingname", "usage_principal"]].copy()
pires = None
pires

In [ ]:
verifier("Exercice 6 · pires contient 5 lignes et la colonne erreur_MkBtu", pires is not None and len(pires) == 5 and "erreur_MkBtu" in pires.columns)
verifier("Exercice 6 · trié par erreur absolue décroissante", lambda: pires["erreur_MkBtu"].abs().is_monotonic_decreasing)

<details><summary>Solution</summary>

```python
erreurs = df.loc[X_test.index, ["buildingname", "usage_principal"]].copy()
erreurs["reel_MkBtu"] = np.expm1(y_test) / 1e6
erreurs["predit_MkBtu"] = np.expm1(pred_test) / 1e6
erreurs["erreur_MkBtu"] = erreurs["predit_MkBtu"] - erreurs["reel_MkBtu"]
pires = erreurs.reindex(erreurs["erreur_MkBtu"].abs().sort_values(ascending=False).index).head(5).round(2)
pires
```
</details>

**Trois enseignements**
1. La **surface** explique l'essentiel, mais c'est l'**usage** qui fait la différence entre deux bâtiments de même taille — un modèle sans catégorie d'usage plafonne.
2. Les pires erreurs en unités réelles sont les **très gros bâtiments** (hôpitaux, campus) : une erreur de 30 % en log devient des dizaines de MkBtu. Pour ces bâtiments, un relevé réel reste indispensable.
3. L'**ENERGY STAR score** apporte une information que rien d'autre ne contient ; sa valeur dépend du coût de le produire.

In [ ]:
print("=== Rapport · section 7 (interprétation) ===")
top3 = pd.Series(np.abs(shap_values.values).mean(axis=0), index=noms_variables).sort_values(ascending=False).head(3)
print("Top 3 variables (|SHAP| moyen) :", {k: round(v, 3) for k, v in top3.items()})
print(comparaison.to_string())
print(f"ENERGY STAR : rang {rang_score}, gain {gain:.0%} de RMSE quand le score est connu")

## 8. Conclusion et recommandation

**Réponse à la question métier** : oui, on peut estimer la consommation d'un bâtiment non résidentiel de Seattle sans relevé, avec un XGBoost réglé qui divise l'erreur de la baseline par deux environ (RMSE en log ≈ 0,4-0,5 contre ≈ 1,0 pour la médiane, R² ≈ 0,8). Surface, usage principal et surface par étage suffisent à expliquer la majorité de la variance. Le score ENERGY STAR améliore la prédiction là où il existe ; le modèle sans score reste le modèle « universel ». Les émissions de CO₂ se prédisent aussi bien, avec les mêmes variables.

**Chiffre clé** : à recopier depuis la cellule Rapport de la section 6 (RMSE test avant → après réglage).

**Limites** : une seule année (2016), une seule ville ; les très gros bâtiments restent mal prédits en valeur absolue ; le nettoyage a retiré ~5 % des bâtiments dont certains étaient peut-être réels ; l'usage principal est déclaratif.

**Avec plus de temps** : ajouter l'année 2015 (plus de données), tester le modèle sur les émissions avec ses propres hyperparamètres, tenter un modèle par grande famille d'usage, calibrer un intervalle de prédiction (quantile regression) pour donner une fourchette plutôt qu'un chiffre.

In [ ]:
MA_SYNTHESE = """
(Remplace ce texte par 5 lignes : ta réponse à la question métier, ton chiffre clé, ta recommandation.)
"""
print(MA_SYNTHESE.strip())

## 9. Pour aller plus loin

- Le dataset complet et les autres années : [Seattle Building Energy Benchmarking (data.seattle.gov)](https://data.seattle.gov/d/teqw-tu6e) · le programme : [Energy Benchmarking — City of Seattle](https://www.seattle.gov/environment/climate-change/buildings-and-energy/energy-benchmarking).
- Tous les hyperparamètres de XGBoost expliqués : [XGBoost Parameters](https://xgboost.readthedocs.io/en/stable/parameter.html) ; pourquoi la recherche aléatoire bat la grille : [Bergstra & Bengio (JMLR 2012)](https://www.jmlr.org/papers/v13/bergstra12a.html).
- SHAP en profondeur (waterfall, interactions, modèles non arborescents) : [documentation SHAP](https://shap.readthedocs.io/en/latest/).
- Optuna à la place de `RandomizedSearchCV` (recherche bayésienne, arrêt précoce) — c'est ce que fait le projet **A4 Scoring crédit**.
- Donner une **fourchette** plutôt qu'un chiffre : [`objective="reg:quantileerror"` dans XGBoost](https://xgboost.readthedocs.io/en/stable/python/examples/quantile_regression.html).
- La brique Le Wagon d'origine (en anglais) : `data-challenges-en/05-ML/07-Ensemble-Methods/01-Houses-Kaggle-Competition` et `08-Projects/04/01-Lecture-Notebook/Explainable_AI_notebook.ipynb`.